# US Post-Market

## 1. Settings

In [1]:
import io
import logging
import os
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, wait
from datetime import datetime, timedelta, timezone
from functools import partial
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
from IPython.display import Markdown, display
from requests.adapters import HTTPAdapter

# 《股市》區塊
INDICES = {"^DJI": "道瓊", "^GSPC": "標普", "^IXIC": "納指", "^SOX": "費半", "^VIX": "VIX"}
ADRS = {"TSM": "台積", "UMC": "聯電", "ASX": "日月光"}           # 台股 ADR
# GICS 類股：維基百科名稱 → (中文, S&P 500 類股指數, 備援 SPDR ETF)
SECTORS = {
    "Information Technology": ("資訊科技", "^SP500-45", "XLK"),
    "Communication Services": ("通訊服務", "^SP500-50", "XLC"),
    "Consumer Discretionary": ("非必需消費", "^SP500-25", "XLY"),
    "Consumer Staples": ("必需消費", "^SP500-30", "XLP"),
    "Energy": ("能源", "^GSPE", "XLE"),             # Yahoo 的能源類股指數代號是 ^GSPE（沒有 ^SP500-10）
    "Financials": ("金融", "^SP500-40", "XLF"),
    "Health Care": ("醫療保健", "^SP500-35", "XLV"),
    "Industrials": ("工業", "^SP500-20", "XLI"),
    "Materials": ("原材料", "^SP500-15", "XLB"),
    "Utilities": ("公用事業", "^SP500-55", "XLU"),
    "Real Estate": ("不動產", "^SP500-60", "XLRE"),
}
FOCUS = {"GS": "Goldman Sachs", "MSFT": "Microsoft", "GOOGL": "Google", "MS": "Morgan Stanley", "AMZN": "Amazon",
         "JPM": "JP Morgan", "META": "Meta", "AAPL": "Apple", "TSLA": "Tesla", "AMD": "AMD", "NVDA": "NVIDIA",
         "SMCI": "SuperMicro"}
ASSETS = {"^TNX": "美債", "DX-Y.NYB": "美元", "GC=F": "黃金", "CL=F": "西德州", "BZ=F": "布蘭特"}
BREADTH = ["RSP", "SPY"]           # 等權重 vs 市值加權
logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # 隱藏 yfinance 的逐檔錯誤訊息，改由第 2 格整理
warnings.filterwarnings("ignore", module=r"yfinance")      # yfinance 內部的 pandas 棄用警告（與資料無關，會洗版）

MOVERS_PER_SECTOR = 3   # 強弱類股各列出幾檔領漲 / 領跌
NEWS_HOURS = 20         # 新聞抓最近幾小時
NEWS_PER_GROUP = 8
SP500_CACHE = "sp500_constituents.csv"
TIMEOUT = 15
YAHOO_WORKERS = 6       # 同時對 Yahoo 發出的請求數（實測 6 沒問題；太多會被限流）
FINNHUB_WORKERS = 8     # Finnhub 免費版：每分鐘 60 次、每秒 30 次
TW = ZoneInfo("Asia/Taipei")
NY = ZoneInfo("America/New_York")
UA = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")}

# 共用連線：連同一個網站多次時不必每次重新建立 HTTPS 連線
HTTP = requests.Session()
HTTP.headers.update(UA)
HTTP.mount("https://", HTTPAdapter(pool_connections=20, pool_maxsize=20))

# 金鑰放在同資料夾的 .env（不要寫進筆記本）
load_dotenv(".env")
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "").strip()   # 個股新聞、財報、綜合新聞
FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()         # 美債殖利率備援
print("Finnhub：", "已設定" if FINNHUB_API_KEY else "未設定", "｜FRED：", "已設定" if FRED_API_KEY else "未設定")


def clean(text, limit=200):
    text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", text or "")).strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "…"


# 不使用的來源（Reuters、Bloomberg 與其他付費牆媒體）：網址或來源名稱符合就略過
PAYWALLED = ("reuters.com", "bloomberg.com", "wsj.com", "ft.com", "barrons.com", "marketwatch.com",
             "economist.com", "nytimes.com", "investors.com", "washingtonpost.com")
PAYWALLED_NAMES = ("reuters", "bloomberg", "wall street journal", "wsj", "financial times", "barron", "marketwatch",
                   "the economist", "new york times", "investor's business daily", "washington post")


def blocked(url="", source=""):
    """Finnhub 的 Reuters 新聞網址是 news.google.com，只看網址擋不掉，所以來源名稱也要比對"""
    u, s = (url or "").lower(), (source or "").lower()
    return any(p in u for p in PAYWALLED) or any(p in s for p in PAYWALLED_NAMES)


def safe_err(e):
    """錯誤訊息遮蔽金鑰"""
    msg = f"{type(e).__name__}: {e}"
    for k in (FINNHUB_API_KEY, FRED_API_KEY):
        if k:
            msg = msg.replace(k, "***")
    return clean(msg, 120)


def attempt(fn, *args, **kwargs):
    """回傳 (結果, 錯誤訊息)：平行抓取時一項失敗不影響其他項"""
    try:
        return fn(*args, **kwargs), None
    except Exception as e:
        return None, safe_err(e)


def gather(tasks, workers, deadline=8):
    """平行執行 tasks = {名稱: (連線池, 無參數函式)}，workers = {連線池: 同時請求數}。
    超過 deadline 秒還沒回來的重送一次（Yahoo 偶爾有請求卡 30 秒才逾時，整格不必跟著等）。
    回傳 {名稱: (結果, 錯誤訊息)}"""
    pools = {p: ThreadPoolExecutor(n) for p, n in workers.items()}
    jobs = {k: pools[p].submit(attempt, fn) for k, (p, fn) in tasks.items()}
    wait(jobs.values(), timeout=deadline)
    for k, f in jobs.items():
        if not f.done():
            p, fn = tasks[k]
            jobs[k] = pools[p].submit(attempt, fn)
    wait(jobs.values(), timeout=deadline)
    for ex in pools.values():
        ex.shutdown(wait=False, cancel_futures=True)     # 卡住的請求留在背景自行逾時
    return {k: f.result() if f.done() else (None, f"逾時（超過 {2 * deadline} 秒）") for k, f in jobs.items()}


FINNHUB_CACHE = {}   # 5 分鐘內重跑同一格直接用上次結果（免費版每分鐘只有 60 次，重跑很容易超過）


def finnhub(path, **params):
    if not FINNHUB_API_KEY:
        raise ValueError("未設定 FINNHUB_API_KEY")
    key = (path, tuple(sorted(params.items())))
    if key in FINNHUB_CACHE and time.time() - FINNHUB_CACHE[key][0] < 300:
        return FINNHUB_CACHE[key][1]
    try:
        for i in range(3):
            # 金鑰放在標頭而不是網址：網址會出現在錯誤訊息裡
            r = HTTP.get(f"https://finnhub.io/api/v1/{path}", params=params,
                         headers={"X-Finnhub-Token": FINNHUB_API_KEY}, timeout=TIMEOUT)
            if r.status_code != 429 or i == 2:
                break
            # 429：第一次多半是每秒上限，等 1.5 秒；再不行就是每分鐘上限，依標頭等到額度重置（最多 5 秒）
            reset = float(r.headers.get("X-Ratelimit-Reset") or 0) - time.time()
            time.sleep(1.5 if i == 0 else min(max(reset + 0.5, 1.5), 5))
        if r.status_code == 429:
            raise RuntimeError("Finnhub 超過免費版每分鐘 60 次上限，約 1 分鐘後重跑本格")
        r.raise_for_status()
        data = r.json()
    except requests.RequestException as e:
        raise RuntimeError(safe_err(e)) from None      # 不論在哪裡被呼叫，錯誤訊息都先遮蔽金鑰
    if isinstance(data, dict) and data.get("error"):
        raise ValueError(data["error"])
    FINNHUB_CACHE[key] = (time.time(), data)
    return data

Finnhub： 已設定 ｜FRED： 已設定


## 2. Prices

In [2]:
try:
    from yfinance.data import YfData    # yfinance 自己的 Yahoo 連線（處理 cookie / crumb）
except ImportError:                     # yfinance 改版找不到時，closes() 全部改用 yf.download
    YfData = None

SPARK = "https://query1.finance.yahoo.com/v7/finance/spark"   # Yahoo 批次端點，一次最多 20 檔
FUTURES = [s for s in ASSETS if s != "^TNX"]   # 幾乎 24 小時交易的黃金、原油、美元指數
FUTURES_CUTOFF = 17                            # 這幾檔的「收盤」＝報告日美東 17:00 前最後一筆價格


def spark_raw(batch, rng, interval):
    """{代號: (時間戳, 收盤序列, meta)}"""
    r = YfData().get(SPARK, params={"symbols": ",".join(batch), "range": rng, "interval": interval}, timeout=TIMEOUT)
    r.raise_for_status()
    out = {}
    for item in r.json()["spark"]["result"] or []:
        resp = (item.get("response") or [{}])[0]
        stamps = resp.get("timestamp") or []
        close = (resp.get("indicators", {}).get("quote") or [{}])[0].get("close") or []
        if stamps and len(stamps) == len(close):
            out[item["symbol"]] = (stamps, close, resp.get("meta") or {})
    return out


def spark(batch):
    """{代號: (近 10 日收盤序列, 第一筆之前的收盤)}，索引是美東日期。
    實測：收盤後幾個小時內，Yahoo 個股 / ETF 的當日日 K 收盤是空值，真正的收盤價放在 meta.regularMarketPrice。
    這裡把它補回去；不補的話當日那筆會被丟掉，整份報告的個股會悄悄變成前一天的數字。"""
    out = {}
    for sym, (stamps, close, meta) in spark_raw(batch, "10d", "1d").items():
        days = pd.to_datetime([datetime.fromtimestamp(x, NY).date() for x in stamps])
        s = pd.Series(close, index=days, dtype=float).dropna()
        s = s[~s.index.duplicated(keep="last")]
        px, t = meta.get("regularMarketPrice"), meta.get("regularMarketTime")
        if px and t:
            day = pd.Timestamp(datetime.fromtimestamp(t, NY).date())
            if s.empty or day > s.index[-1]:
                s[day] = float(px)
        if not s.empty:
            out[sym] = (s, meta.get("chartPreviousClose"))
    return out


def futures_close(symbols, day):
    """{代號: 報告日美東 17:00 前最後一筆價格}（30 分 K，一次請求）。
    期貨的當日日 K 在美股收盤後會變成夜盤即時價，晚一點重跑數字就不一樣；固定截在 17:00，什麼時候跑都相同。
    清晨 5–6 點（美東 16–17 點）執行時，這就是當下的最新價。"""
    end = datetime.combine(day, datetime.min.time(), NY).replace(hour=FUTURES_CUTOFF).timestamp()
    out = {}
    for sym, (stamps, close, _) in spark_raw(symbols, "5d", "30m").items():
        v = [c for t, c in zip(stamps, close) if end - FUTURES_CUTOFF * 3600 <= t < end and c is not None]
        if v:
            out[sym] = float(v[-1])
    return out


def closes(symbols):
    """近 10 個交易日收盤價（列：美東交易日，欄：代號）。
    先用 Yahoo 批次端點（20 檔一組、多組同時抓，500 檔約 2–3 秒）；抓不到的再用 yf.download 補（較慢）。"""
    symbols = list(dict.fromkeys(symbols))
    data = {}
    if YfData is not None:
        batches = [symbols[i:i + 20] for i in range(0, len(symbols), 20)]
        res = gather({i: ("yahoo", partial(spark, b)) for i, b in enumerate(batches)}, {"yahoo": YAHOO_WORKERS})
        for part, _ in res.values():
            data.update(part or {})
    wanted = set(symbols)
    c = pd.DataFrame({s: v[0] for s, v in data.items() if s in wanted})
    missing = [s for s in symbols if s not in c]
    if missing:
        df = yf.download(missing, period="10d", interval="1d", auto_adjust=False, progress=False)
        if not df.empty and "Close" in df:
            extra = df["Close"]
            extra = extra.to_frame(missing[0]) if isinstance(extra, pd.Series) else extra
            c = c.join(extra.dropna(axis=1, how="all"), how="outer") if not c.empty else extra
    if c.empty:
        raise RuntimeError("Yahoo Finance 沒有回傳資料（請檢查網路）")
    c = c.sort_index()
    c.attrs["prev_close"] = {s: v[1] for s, v in data.items()}
    return c


def last_two(c, as_of):
    """{代號: (報告日收盤, 前一日收盤)}。
    沒有報告日當天價格的代號不放進來（會列在「缺資料」）：寧可缺，也不拿前一天的數字頂替。
    Yahoo 只給最新一筆日 K 的代號（S&P 500 類股指數），用該筆之前的收盤當前一日收盤。"""
    prev0 = c.attrs.get("prev_close", {})
    out = {}
    for s, col in c.items():
        v = col.dropna().loc[:as_of]
        if v.empty or v.index[-1] != as_of:
            continue
        if len(v) >= 2:
            out[s] = (float(v.iloc[-1]), float(v.iloc[-2]))
        elif prev0.get(s):
            out[s] = (float(v.iloc[-1]), float(prev0[s]))
    return out


def chg(s, q=None):
    last, prev = (q or Q)[s]
    return last - prev, (last / prev - 1) * 100, last


symbols = [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] + [x for _, idx, etf in SECTORS.values() for x in (idx, etf) if x]
C = closes(symbols)
AS_OF = C[[s for s in INDICES if s in C]].dropna(how="all").index[-1]
fut, fut_err = attempt(futures_close, [s for s in FUTURES if s in C], AS_OF.date())
for s, px in (fut or {}).items():
    C.loc[AS_OF, s] = px
Q = last_two(C, AS_OF)
# 類股漲跌：優先用 S&P 500 類股指數，沒有資料才用 SPDR ETF
SECTOR_PCT, via_etf, no_sector = {}, [], []
for en, (zh, idx, etf) in SECTORS.items():
    s = idx if idx in Q else etf if etf in Q else None
    if s:
        SECTOR_PCT[en] = chg(s)[1]
        if s == etf:
            via_etf.append(f"{zh}（{etf}）")
    else:
        no_sector.append(zh)
RANKED = sorted(SECTOR_PCT, key=SECTOR_PCT.get, reverse=True)
missing = [s for s in [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] if s not in Q] + no_sector
now = datetime.now(NY)
print(f"資料日期：{AS_OF:%Y-%m-%d}（美東收盤）｜執行時間：台北 {now.astimezone(TW):%m/%d %H:%M}＝美東 {now:%m/%d %H:%M}")
if now.date() == AS_OF.date() and now.hour < 16:
    print("⚠️ 美股還沒收盤（美東 16:00），以下是盤中數字，不是收盤價")
no_cut = [s for s in FUTURES if s in Q and s not in (fut or {})]
if no_cut:
    print(f"⚠️ {'、'.join(ASSETS[s] for s in no_cut)} 抓不到美東 {FUTURES_CUTOFF}:00 的價格"
          f"{f'（{fut_err}）' if fut_err else ''}，改用日線最新價（美東 18:00 後執行會是夜盤價）")
print("類股改用 ETF：" + "、".join(via_etf) if via_etf else "類股全部使用 S&P 500 類股指數")
print("缺資料（沒有報告日當天的價格，稍後重跑本格）：" + "、".join(missing) if missing else "全部抓取成功")

資料日期：2026-11-11（美東收盤）｜執行時間：台北 11/12 09:36＝美東 11/11 20:36
類股全部使用 S&P 500 類股指數
全部抓取成功


## 3. Report Data Blocks

In [3]:
def updown(d, digits=2):
    return f"{'漲' if d >= 0 else '跌'}{abs(d):.{digits}f}"


out = ["《股市》", ""]
for s, name in {**INDICES, **ADRS}.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
        continue
    d, p, last = chg(s)
    if s == "^VIX":
        out += [f"🔘{name} {updown(d, 1)} ({p:+.2f}%) 收 {last:.2f}", ""]
    elif s in INDICES:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.0f}", ""]
    else:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.2f}", ""]
out += ["（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）", "", "（類股評論：最強類股 → 次強 → 最弱，各一段）", ""]
out += ["🔹強勢類股：", ""] + [x for s in RANKED[:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
out += ["🔹弱勢類股：", ""] + [x for s in RANKED[::-1][:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
focus = sorted((s for s in FOCUS if s in Q), key=lambda s: chg(s)[1], reverse=True)
out += ["🔹焦點股： " + "、".join(f"{FOCUS[s]} {chg(s)[1]:+.2f}%" for s in focus), ""]
out += ["《美債/美元/黃金/原油》", ""]
for s, name in ASSETS.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
    elif s == "^TNX":
        last, prev = Q[s]
        out += [f"🔘{name} {round((last - prev) * 100):+d}bps 收{last:.2f}%", ""]
    else:
        d, p, last = chg(s)
        out += [f"🔘{name} {p:+.2f}% 收 {last:.2f}", ""]
out += [".", ""] + [f"🔹{k}：（待撰寫）\n" for k in ("美債", "美元", "黃金", "原油")]
out += ["《重點訊息》", "", "🔹利多：", "", "1. （待撰寫）", "", "🔹利空：", "", "1. （待撰寫）"]
print("\n".join(out))

《股市》

🔘道瓊 漲254.60 (+0.49%) 收 52486

🔘標普 漲98.45 (+1.31%) 收 7619

🔘納指 漲400.32 (+1.60%) 收 25466

🔘費半 漲565.78 (+5.66%) 收 10564

🔘VIX 跌2.0 (-10.12%) 收 17.49

🔘台積 漲16.87 (+4.25%) 收 413.35

🔘聯電 漲1.04 (+6.91%) 收 16.08

🔘日月光 漲2.34 (+6.60%) 收 37.82

（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）

（類股評論：最強類股 → 次強 → 最弱，各一段）

🔹強勢類股：

資訊科技 +2.71%

能源 +1.83%

工業 +1.32%

🔹弱勢類股：

非必需消費 -0.36%

不動產 +0.03%

醫療保健 +0.06%

🔹焦點股： SuperMicro +9.50%、AMD +7.32%、Tesla +2.91%、NVIDIA +2.87%、Google +2.11%、Apple +2.04%、Goldman Sachs +1.83%、Morgan Stanley +1.63%、JP Morgan +0.39%、Amazon -1.74%、Meta -1.97%、Microsoft -2.26%

《美債/美元/黃金/原油》

🔘美債 +2bps 收5.00%

🔘美元 +0.67% 收 102.91

🔘黃金 -0.89% 收 3994.30

🔘西德州 +0.44% 收 86.42

🔘布蘭特 +1.25% 收 92.63

.

🔹美債：（待撰寫）

🔹美元：（待撰寫）

🔹黃金：（待撰寫）

🔹原油：（待撰寫）

《重點訊息》

🔹利多：

1. （待撰寫）

🔹利空：

1. （待撰寫）


## 4. Sector detail, top movers, market breadth

In [4]:
def sp500():
    """S&P 500 成分股（維基百科），快取 7 天"""
    if os.path.exists(SP500_CACHE) and time.time() - os.path.getmtime(SP500_CACHE) < 7 * 86400:
        return pd.read_csv(SP500_CACHE)
    r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    t = next(t for t in pd.read_html(io.StringIO(r.text)) if {"Symbol", "GICS Sector"} <= set(t.columns))
    df = t.rename(columns={"Symbol": "symbol", "Security": "name", "GICS Sector": "sector",
                           "GICS Sub-Industry": "industry"})[["symbol", "name", "sector", "industry"]]
    df["symbol"] = df["symbol"].astype(str).str.replace(".", "-", regex=False)   # BRK.B → BRK-B
    df.to_csv(SP500_CACHE, index=False)
    return df


def tag(r):
    return f"{r['name']}（{r['symbol']} {r['chg']:+.2f}%）"


UNI = sp500()
QU = last_two(closes(UNI["symbol"]), AS_OF)
UNI["chg"] = [chg(s, QU)[1] if s in QU else None for s in UNI["symbol"]]
n_missing = int(UNI["chg"].isna().sum())
UNI = UNI.dropna(subset=["chg"])

adv, dec = int((UNI["chg"] > 0).sum()), int((UNI["chg"] < 0).sum())
lines = [f"**市場廣度**：上漲 {adv} 家、下跌 {dec} 家（漲跌家數比 {adv / max(dec, 1):.2f}）"]
if all(s in Q for s in BREADTH):
    lines[0] += f"｜等權重 RSP {chg('RSP')[1]:+.2f}% vs 市值加權 SPY {chg('SPY')[1]:+.2f}%"
if n_missing:
    lines[0] += f"｜⚠️ {n_missing} 檔沒有報告日當天的價格，未計入"
MOVERS = []
for en in RANKED[:3] + RANKED[::-1][:3]:
    sub = UNI[UNI["sector"] == en].sort_values("chg", ascending=False)
    if sub.empty:
        continue
    ind = sub.groupby("industry")["chg"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    fmt_ind = lambda rows: "、".join(f"{i} {m:+.2f}%（{int(n)}檔）" for i, (m, n) in rows.iterrows()) or "—"
    top = sub[sub["chg"] > 0].head(MOVERS_PER_SECTOR)
    bottom = sub[sub["chg"] < 0].tail(MOVERS_PER_SECTOR).iloc[::-1]
    for _, r in pd.concat([top, bottom]).iterrows():
        if r["symbol"] not in {m["symbol"] for m in MOVERS}:
            MOVERS.append(r)
    lines += ["", f"### {SECTORS[en][0]} {SECTOR_PCT[en]:+.2f}%",
              f"- 子產業最強：{fmt_ind(ind[ind['mean'] > 0].head(3))}",
              f"- 子產業最弱：{fmt_ind(ind[ind['mean'] < 0].tail(3).iloc[::-1])}",
              f"- 領漲：{'、'.join(tag(r) for _, r in top.iterrows()) or '—'}",
              f"- 領跌：{'、'.join(tag(r) for _, r in bottom.iterrows()) or '—'}"]
display(Markdown("\n".join(lines)))

**市場廣度**：上漲 332 家、下跌 168 家（漲跌家數比 1.98）｜等權重 RSP +0.80% vs 市值加權 SPY +1.32%

### 資訊科技 +2.71%
- 子產業最強：Technology Hardware, Storage & Peripherals +7.05%（10檔）、Semiconductor Materials & Equipment +5.61%（5檔）、Electronic Components +5.29%（3檔）
- 子產業最弱：—
- 領漲：Sandisk（SNDK +13.69%）、Micron Technology（MU +11.89%）、Dell Technologies（DELL +11.75%）
- 領跌：Microsoft（MSFT -2.26%）、Accenture（ACN -1.88%）、First Solar（FSLR -1.59%）

### 能源 +1.83%
- 子產業最強：Oil & Gas Refining & Marketing +3.43%（3檔）、Oil & Gas Equipment & Services +2.97%（3檔）、Oil & Gas Storage & Transportation +2.02%（4檔）
- 子產業最弱：—
- 領漲：Targa Resources（TRGP +3.72%）、Valero Energy（VLO +3.71%）、APA Corporation（APA +3.59%）
- 領跌：—

### 工業 +1.32%
- 子產業最強：Heavy Electrical Equipment +4.06%（2檔）、Electrical Components & Equipment +3.95%（6檔）、Construction & Engineering +3.21%（4檔）
- 子產業最弱：Passenger Airlines -0.81%（3檔）、Diversified Support Services -0.65%（3檔）、Industrial Conglomerates -0.44%（3檔）
- 領漲：Bloom Energy（BE +12.97%）、Generac（GNRC +6.72%）、Vertiv（VRT +6.33%）
- 領跌：Leidos（LDOS -2.44%）、United Airlines Holdings（UAL -1.82%）、Xylem Inc.（XYL -1.79%）

### 非必需消費 -0.36%
- 子產業最強：Consumer Electronics +1.83%（1檔）、Leisure Products +1.81%（1檔）、Automotive Retail +1.03%（3檔）
- 子產業最弱：Apparel, Accessories & Luxury Goods -1.66%（4檔）、Footwear -1.57%（1檔）、Other Specialty Retail -1.41%（2檔）
- 領漲：Tesla, Inc.（TSLA +2.91%）、Expedia Group（EXPE +2.37%）、Chipotle Mexican Grill（CMG +2.00%）
- 領跌：Airbnb（ABNB -3.09%）、Lululemon Athletica（LULU -2.65%）、McDonald's（MCD -2.61%）

### 不動產 +0.03%
- 子產業最強：Other Specialized REITs +1.49%（1檔）、Data Center REITs +0.93%（2檔）、Office REITs +0.35%（2檔）
- 子產業最弱：Multi-Family Residential REITs -0.80%（5檔）、Single-Family Residential REITs -0.78%（1檔）、Hotel & Resort REITs -0.39%（2檔）
- 領漲：Equinix（EQIX +1.72%）、Iron Mountain（IRM +1.49%）、Alexandria Real Estate Equities（ARE +0.85%）
- 領跌：Vivmark Residential（VMRK -1.70%）、Camden Property Trust（CPT -0.80%）、Invitation Homes（INVH -0.78%）

### 醫療保健 +0.06%
- 子產業最強：Biotechnology +0.72%（8檔）、Health Care Services +0.57%（5檔）、Health Care Facilities +0.51%（2檔）
- 子產業最弱：Health Care Technology -0.94%（2檔）、Health Care Supplies -0.73%（3檔）、Life Sciences Tools & Services -0.46%（9檔）
- 領漲：Regeneron Pharmaceuticals（REGN +4.73%）、DaVita（DVA +2.35%）、ResMed（RMD +1.89%）
- 領跌：Stryker Corporation（SYK -1.70%）、Solventum（SOLV -1.62%）、GE HealthCare（GEHC -1.26%）

## 5. Why stocks moved: analyst ratings + headlines

In [5]:
ACTION = {"up": "升評", "down": "降評", "init": "首評", "main": "維持", "reit": "重申"}
DAY = AS_OF.date()
HOUR = {"bmo": "盤前", "amc": "盤後", "dmh": "盤中"}


def analyst(sym, days=3):
    """Yahoo 分析師評等變動（近 N 天）"""
    ud = yf.Ticker(sym).upgrades_downgrades
    if ud is None or ud.empty:
        return []
    idx = pd.to_datetime(ud.index)
    idx = idx.tz_localize(None) if idx.tz is not None else idx
    ud = ud[idx >= pd.Timestamp(AS_OF) - pd.Timedelta(days=days)]
    out = []
    for _, r in ud.iterrows():
        text = f"{r.get('Firm', '')} {ACTION.get(r.get('Action'), r.get('Action', ''))}"
        if r.get("FromGrade") and r.get("FromGrade") != r.get("ToGrade"):
            text += f"：{r['FromGrade']} → {r.get('ToGrade', '')}"
        elif r.get("ToGrade"):
            text += f"：{r['ToGrade']}"
        if pd.notna(r.get("currentPriceTarget")) and r.get("currentPriceTarget"):
            prior = r.get("priorPriceTarget")
            text += f"，目標價 {prior:g} → {r['currentPriceTarget']:g}" if pd.notna(prior) and prior else \
                f"，目標價 {r['currentPriceTarget']:g}"
        out.append(text)
    return out


def stock_news(sym, n=3):
    """有 Finnhub 金鑰用 Finnhub（含摘要），否則用 Yahoo"""
    if FINNHUB_API_KEY:
        data = finnhub("company-news", symbol=sym, **{"from": f"{DAY - timedelta(days=1):%Y-%m-%d}",
                                                       "to": f"{DAY + timedelta(days=1):%Y-%m-%d}"})
        data = sorted((d for d in data if d.get("headline") and not blocked(d.get("url"), d.get("source"))),
                      key=lambda d: d.get("datetime", 0), reverse=True)[:n]
        return [f"[{clean(d['headline'], 200)}]({d.get('url', '')}) — {d.get('source', '')}"
                + (f"：{clean(d['summary'], 160)}" if d.get("summary") else "") for d in data]
    out = []
    for it in (yf.Ticker(sym).news or [])[:n]:
        c = it.get("content", it)            # yfinance 新舊兩種格式
        url = (c.get("canonicalUrl") or {}).get("url") or c.get("link", "")
        src = (c.get("provider") or {}).get("displayName") or c.get("publisher", "")
        if c.get("title"):
            out.append(f"[{c['title']}]({url}) — {src}")
    return out


# ---- 當日財報（Finnhub）：寫「重點訊息」與「盤後」段落用 ----
def beat(actual, est):
    if actual is None or est is None:
        return ""
    return "（優於預期）" if actual > est else "（不如預期）" if actual < est else "（符合預期）"


def earnings_table(data):
    sp = UNI.set_index("symbol")
    cal = sorted((d for d in data["earningsCalendar"] if d.get("symbol") in sp.index),
                 key=lambda d: -(d.get("revenueEstimate") or 0))
    if not cal:
        return "- 當日無 S&P 500 成分股公布財報"
    rev = lambda v: f"{v / 1e8:,.1f}" if v else "—"
    eps = lambda v: f"{v:.2f}" if v is not None else "—"
    rows = ["| 公司 | 時段 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 當日股價 |", "|:---|:---|---:|---:|---:|"]
    for d in cal[:20]:
        px = f"{sp.loc[d['symbol'], 'chg']:+.2f}%" if pd.notna(sp.loc[d["symbol"], "chg"]) else "—"
        rows.append(f"| {sp.loc[d['symbol'], 'name']}（{d['symbol']}） | {HOUR.get(d.get('hour'), d.get('hour') or '—')} | "
                    f"{eps(d.get('epsActual'))} / {eps(d.get('epsEstimate'))}{beat(d.get('epsActual'), d.get('epsEstimate'))} | "
                    f"{rev(d.get('revenueActual'))} / {rev(d.get('revenueEstimate'))} | {px} |")
    return "\n".join(rows) + "\n\n盤後公布的財報，當日股價尚未反映，實際數字也可能稍晚才更新。"


# 評等（Yahoo）、新聞與財報（Finnhub）分兩組同時抓
tasks = {"cal": ("finnhub", partial(finnhub, "calendar/earnings", **{"from": f"{DAY:%Y-%m-%d}", "to": f"{DAY:%Y-%m-%d}"}))}
for r in MOVERS:
    tasks["analyst", r["symbol"]] = ("yahoo", partial(analyst, r["symbol"]))
    tasks["news", r["symbol"]] = ("finnhub", partial(stock_news, r["symbol"]))
res = gather(tasks, {"yahoo": YAHOO_WORKERS,
                     "finnhub": FINNHUB_WORKERS if FINNHUB_API_KEY else 2})   # 沒金鑰時新聞也走 Yahoo，壓低同時請求數

lines, analyst_errs = [], []
for r in MOVERS:
    (ratings, ratings_err), (news, news_err) = res["analyst", r["symbol"]], res["news", r["symbol"]]
    lines.append(f"**{tag(r)}**")
    lines += [f"- 📊 {a}" for a in ratings or []]
    lines += [f"- 📰 {n}" for n in news or []]
    if news_err:
        lines.append(f"- 📰 （新聞無法取得：{news_err}）")
    if ratings_err:
        analyst_errs.append(ratings_err)
    if not ratings and not news and not news_err:
        lines.append("- （近期無評等變動或新聞）")
    lines.append("")
if analyst_errs:
    lines.append(f"⚠️ 分析師評等 {len(analyst_errs)}/{len(MOVERS)} 檔無法取得（Yahoo 限流時常見，稍後重跑本格即可）：{analyst_errs[0]}")
display(Markdown("\n".join(lines)))

cal, err = res["cal"]
body, err = (None, err) if err else attempt(earnings_table, cal)
display(Markdown(f"**當日財報｜{DAY:%Y-%m-%d}（S&P 500 成分股，依營收規模排序）**\n\n{body or f'- 無法取得：{err}'}"))

**Sandisk（SNDK +13.69%）**
- 📰 [Dow Jones Futures: Techs Tumble Late As Nvidia May Oust CEO Huang; AI Stocks Slide, But Not AMD](https://finnhub.io/api/news?id=999ed9adfd53008a0f30252393df8c949ff7c3ca3fdfad8c9128ced5564281b0) — Yahoo：Techs fell as Nvidia dived on news that it may oust CEO Jensen Huang. AMD extended gains. Will that spoil Wednesday's rally?
- 📰 [Analyst Says Sandisk Corp. (SNDK) Offers More Than 100% Upside. Should You Buy?](https://finnhub.io/api/news?id=9e3235b9b7a6ef67d1a7254fe9ec87523d75e978cf886d1f16511f6a70b0b982) — Yahoo：Sandisk Corp. (NASDAQ:SNDK) is on the AI Stocks Dominating List of Biggest Winners. Sandisk saw its share price climb by 13.69 percent on Wednesday to end at $9…
- 📰 [Jim Cramer Says Micron Isn’t Going Away. Here’s Why He Picks It Over Sandisk](https://finnhub.io/api/news?id=b357db596a5071e2dd541a8a3fd2ad7b227fa3b55b5357822ec675b8aeae8e76) — Yahoo：Memory stocks just ripped higher, yet both names remain deep in the red for the month. Cramer sees one clear winner in the group, but the bull case for its riva…

**Micron Technology（MU +11.89%）**
- 📰 [Stock Market Today, Nov. 11: Stock Market Today, Nov. 11: Nasdaq Surges as CoreWeave's 25% Rally Reignites the AI Trade](https://finnhub.io/api/news?id=b3d97c9cdb2a2c151bfb29be44fc785c59d63530cdd8f2c6cc2a0dda2d6bfac4) — Yahoo：Wall Street staged a strong recovery Wednesday as enthusiasm for AI infrastructure returned, lifting semiconductor and technology stocks despite persistent conc…
- 📰 [Micron Stock Is Down 40% From Its Peak After the Sell-Off. History Says This Is What Happens Next.](https://finnhub.io/api/news?id=167e20fc204925e5ce6aaf10c261b8bb8f38dfe385b22abf090489040fa0b56e) — Yahoo：Micron can go on a terrific bull run once again amid favorable conditions in the memory market.
- 📰 [How Much Tax Does a 66-Year-Old Owe Selling $600,000 in Micron and Oracle Gains?](https://finnhub.io/api/news?id=eb90f1b48efa7a8fb7eb9e3433b147104517dbf317b585cc9762e5e77b35d67f) — Yahoo：A retired couple sitting on $600,000 of Micron and Oracle gains can take it all now or stretch it out. One path cuts the tax bill but triggers a surprise Medica…

**Dell Technologies（DELL +11.75%）**
- 📰 [Is Dell Stock Priced Right Against Its Peers?](https://finnhub.io/api/news?id=825afe4818a1adcb767552dc041f4b23716414a3d3407e5e949b66fb6aa3f5f0) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [S&P 500 Top Movers: Nov. 11, 2026 Session Gainers and Losers](https://finnhub.io/api/news?id=b6edd76c9f28a0250a1d0e93dd4e23366891419b67e1158365a3fb327561a645) — ChartMill：S&P 500 top movers for the Nov. 11, 2026 US session, including gainers SNDK, BE, MU, DELL and STX, and losers BF.B, ABNB, CLX, MSFT and LDOS.
- 📰 [Dell Stocks Jump 11.42% as CoreWeave's Bigger Spending Plan Revives AI Server Bets](https://finnhub.io/api/news?id=2db68ca97702926477d0dad0b672bd63502e5490fe862991186e80448529a23e) — Yahoo：CoreWeave lifted the low end of its 2026 capex range to $37 billion, a read-through for Dell's record $95 billion AI backlog.

**Microsoft（MSFT -2.26%）**
- 📰 [Fusion's Next Race Is Over Superconducting Tape, Not Plasma](https://finnhub.io/api/news?id=5deaac082d38d7e5742c95f8543b43b792bc36f8e813b5ab3b043c25f0219979) — Yahoo：Proxima Fusion signed deals with Japan's Fujikura and Kyoto Fusioneering to build a European fusion supply chain, starting with superconducting tape.
- 📰 [Stock Market Today, Nov. 11: IREN Rises as AI Cloud Peers Rally on CoreWeave Results](https://finnhub.io/api/news?id=1230866a7d1728627bbf032a179e321a9f98575dfa21c1328f5582a47d65f5b4) — Yahoo：Trading volume ran well above its three-month average as the AI data center operator rode a sector rally, today, Nov. 11, 2026.
- 📰 [Stocks Slide as AI Trade Roars Back | Closing Bell](https://finnhub.io/api/news?id=3d479acb3619be1e4dec13615e1e66119b302189ffe792bf42e2b70da5c9cc3e) — Yahoo：Comprehensive cross-platform coverage of the U.S. market close on Bloomberg Television, Bloomberg Radio, and YouTube with Romaine Bostick, Vonnie Quinn, Carol M…

**Accenture（ACN -1.88%）**
- 📰 [What Will Decide Accenture Stock Over The Next Six Months?](https://finnhub.io/api/news?id=7d5b515d293a1e181ab3b1f549bbdca48f1413cc5c09e03559dfa99330cb1190) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [S&P 500 Top Movers: Nov. 11, 2026 Session Gainers and Losers](https://finnhub.io/api/news?id=b6edd76c9f28a0250a1d0e93dd4e23366891419b67e1158365a3fb327561a645) — ChartMill：S&P 500 top movers for the Nov. 11, 2026 US session, including gainers SNDK, BE, MU, DELL and STX, and losers BF.B, ABNB, CLX, MSFT and LDOS.
- 📰 [SpaceX’s Starlink Sees Strong Subscriber Growth in H1 2026, But Ookla Asks Whether Network Capacity Can Keep Up With Growing Demand](https://finnhub.io/api/news?id=2dac6b3db4837c5be1cff4c1a40e916f183d92406dd16e91ab4f218949d7a357) — Yahoo：Ookla highlighted that Starlink’s median download speeds declined in 16 states and Washington, D.C in the first half of 2026.

**First Solar（FSLR -1.59%）**
- 📊 UBS 維持：Buy，目標價 330 → 295
- 📊 Piper Sandler 維持：Overweight，目標價 251 → 222
- 📰 [First Solar Stock: Is Wall Street Bullish or Bearish?](https://finnhub.io/api/news?id=a8df67bdf9de8bbed65ba3fd8b11b4c1983e4efd2b8bccf3792434ce2a635d7e) — Yahoo：Despite underperforming the broader market over the past year, First Solar continues to receive a cautiously optimistic outlook from Wall Street analysts.
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.

**Targa Resources（TRGP +3.72%）**
- 📊 Morgan Stanley 維持：Overweight，目標價 359 → 362
- 📊 Scotiabank 維持：Sector Outperform，目標價 319 → 325
- 📊 Wells Fargo 維持：Overweight，目標價 324 → 330
- 📰 [TRGP Q3 Earnings Beat on Permian Growth and Higher Fees](https://finnhub.io/api/news?id=b3c098f665be93468a3d60ffd9478748bf2b4db862566c78416fefb033ab0fb5) — Yahoo：Targa Resources' Q3 2026 earnings beat estimates as record Permian volumes and higher fees fueled growth, prompting a raised full-year EBITDA outlook.
- 📰 [Is Rising U.S. Natural Gas Demand Fueling a Pipeline Investment Boom?](https://finnhub.io/api/news?id=828418bc05d04af4427e2259d4f15550c0238ee81dcc7f1b6e42007d750d451e) — Yahoo：Rising LNG exports and electricity demand fuel U.S. natural gas pipeline expansion as Kinder Morgan, Williams and Targa advance major infrastructure investments…
- 📰 [Elite Dividend Growth Machines On Sale (Up To 7% Yields)](https://finnhub.io/api/news?id=e0b413cfbb2325fe7b408b197c1186b28feb53870c7e13d7fe105bea1d405a89) — SeekingAlpha：The bull case for two dividend funds: 4.4% and 7.4% yields, long-term distribution growth and exposure to AI and rising electricity demand. Read more here.

**Valero Energy（VLO +3.71%）**
- 📰 [Will Wider Heavy Crude Discounts Strengthen Phillips 66's Margins?](https://finnhub.io/api/news?id=6a2ae3b07640205edddb55fe382a8567bc179f17f5f3e39dbdd9016a7754aeeb) — Yahoo：PSX stands to benefit from widening heavy crude discounts, leveraging its vast processing capacity and pipeline network to boost margins and profitability.
- 📰 [3 Reasons Why Growth Investors Shouldn't Overlook Valero Energy (VLO)](https://finnhub.io/api/news?id=1f012c9d5fd532de3ea1c66671e98876d20951b22218b2f9f995deb368742ea0) — Yahoo：Valero Energy (VLO) is well positioned to outperform the market, as it exhibits above-average growth in financials.
- 📰 [PBF Energy Is Now Undervalued: Is This a Strong Entry Point?](https://finnhub.io/api/news?id=c8f284dbeb31c38e56b67d4c6515d1c82e70f3d07ab6017d3ff9b525684fe46c) — Yahoo：PBF trades at a steep discount to industry peers, while favorable refining margins, renewable diesel growth and a strong balance sheet bolster its outlook.

**APA Corporation（APA +3.59%）**
- 📊 Truist Securities 維持：Hold，目標價 45 → 50
- 📰 [Should Value Investors Buy APA (APA) Stock?](https://finnhub.io/api/news?id=5066903b07124c963755e2d97b283735f5b1829632ae685b9c718c11e1767eb8) — Yahoo：Here at Zacks, our focus is on the proven Zacks Rank system, which emphasizes earnings estimates and estimate revisions to find great stocks. Nevertheless, we a…
- 📰 [APA Raises 2026 Free Cash Flow Forecast to $2.9B After Q3 Results](https://finnhub.io/api/news?id=5e954384fa04455b4c472dd22d96a9768de9ba5d95a46b2d839468e18bcf04e9) — Yahoo：APA raises its 2026 free cash flow outlook to $2.9 billion as stronger U.S. oil production, higher prices and cost controls boost shareholder returns and debt r…
- 📰 [EIA Sees Brent Near $100 for the Rest of the Year as Crude Futures Ease Again](https://finnhub.io/api/news?id=86b5d77d61d1a53c929950879377b6f35a5e2a176353c3e948791ae83de6b647) — Yahoo：Oil Market Daily News Commentary - U.S. crude oil futures closed lower for a second straight session on Tuesday. The December NYMEX West Texas Intermediate cont…

**Bloom Energy（BE +12.97%）**
- 📊 RBC Capital 重申：Outperform，目標價 335 → 335
- 📰 [Dow Jones Futures: Techs Tumble Late As Nvidia May Oust CEO Huang; AI Stocks Slide, But Not AMD](https://finnhub.io/api/news?id=999ed9adfd53008a0f30252393df8c949ff7c3ca3fdfad8c9128ced5564281b0) — Yahoo：Techs fell as Nvidia dived on news that it may oust CEO Jensen Huang. AMD extended gains. Will that spoil Wednesday's rally?
- 📰 [What’s Propelling Bloom Energy’s (BE) 13% Run?](https://finnhub.io/api/news?id=5c819833e358e11a47bb22c28df93252d3a0cd87f94a915cdc88bc2adcacbb03) — Yahoo：Bloom Energy Corp. (NYSE:BE) is one of the AI Stocks Dominating List of Biggest Winners. Bloom Energy saw its share price increase by 12.99 percent on Wednesday…
- 📰 [Bloom Energy Just Jumped 13% After CoreWeave Calmed AI Data Center Fears. History Says What Usually Comes Next for a Stock That Halved](https://finnhub.io/api/news?id=f4dac49b27f7101d0e06ce8e0768668960bf3eaf7cf1617700092d08a4b6eb01) — Yahoo：Bloom Energy jumped 13% on Wednesday after CoreWeave's results eased fears about AI data center spending, adding about $7 billion of market value. The stock is…

**Generac（GNRC +6.72%）**
- 📊 Piper Sandler 維持：Overweight，目標價 279 → 266
- 📰 [Generac: 3 Key Takeaways for Long-Term Investors](https://finnhub.io/api/news?id=c20957cbd6c05c19d7b1dec3cb6e6a66fce60f92090cfbfafa7ae863a92fd2d5) — Yahoo：Manufacturing is working to triple production with minimal capital spending, a structural advantage that reshapes return-on-investment expectations.
- 📰 [AI Data Center Suppliers Report Growth as October Inflation Data Reaches Wall Street](https://finnhub.io/api/news?id=dfa16c625436befa4d3f62e9a7d35dc45c7bfb9c1691a5c7a3551b1176a7c5bd) — Yahoo：Equity Insider News Commentary - U.S. stocks were lower in late morning trading on Tuesday, the day the Bureau of Labor Statistics published its October Consume…
- 📰 [Stem Acquires Software Assets Of raicoon For Automated Fault Detection And Workflow Management In Solar Portfolios; Terms Not Disclosed](https://finnhub.io/api/news?id=16e82977bee56ffad394c13a7ee4aabbb77846e2aaedc180df67a2bfb4501f9c) — Benzinga

**Vertiv（VRT +6.33%）**
- 📰 [The Reason To Buy Vertiv Stock Today](https://finnhub.io/api/news?id=c254db10484d1f1d39e2723c7675e76d55f84be96420459d8e4a0b71d0d2b2a9) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [Vertiv Stocks Climb 6.9% as AI Data Center Rebound Lifts Power and Cooling Supplier](https://finnhub.io/api/news?id=d163d80f942a2e3c443e201e6fa69df73d727b0c1043a422392ed7f47bf9d287) — Yahoo：Vertiv's organic orders more than doubled last quarter, and full-year sales guidance rose by $650 million.
- 📰 [FPS Declines 42% in a Month: Time to Buy the Stock Now?](https://finnhub.io/api/news?id=d8429ef38c7460cbd93643245377852559ecf1c9ed4c2746c3b596398fc1b7c5) — Yahoo：Forgent Power Solutions combines surging data center demand, record orders and expanding production capacity.

**Leidos（LDOS -2.44%）**
- 📊 Citigroup 維持：Buy，目標價 141 → 163
- 📰 [S&P 500 Top Movers: Nov. 11, 2026 Session Gainers and Losers](https://finnhub.io/api/news?id=b6edd76c9f28a0250a1d0e93dd4e23366891419b67e1158365a3fb327561a645) — ChartMill：S&P 500 top movers for the Nov. 11, 2026 US session, including gainers SNDK, BE, MU, DELL and STX, and losers BF.B, ABNB, CLX, MSFT and LDOS.
- 📰 [S&P 500 Top Movers Today: Double-Digit Gainers Lead, Declines Stay Smaller](https://finnhub.io/api/news?id=2b884a383f63ba74edf4a45050a46bf32c23166fd573ba68c9077445c2089631) — ChartMill：S&P 500 top movers for today's session: Bloom Energy and Sandisk led double-digit gainers, while Clorox and Lululemon paced much smaller declines.
- 📰 [One Reason To Buy Planet Labs Stock](https://finnhub.io/api/news?id=60944b17a25ce8d30efc75ef67087ce452433ff100f4c74c99180440d0bbb5e8) — Yahoo：For questions about this article or underlying research, email help@trefis.com.

**United Airlines Holdings（UAL -1.82%）**
- 📰 [Sector Update: Consumer Stocks Mixed Late Afternoon](https://finnhub.io/api/news?id=3a665a79fa89338ab9da0e4c65db9e9542a5b986f6df7471a81a297e36eaafe3) — Yahoo：Consumer stocks were mixed late Wednesday afternoon, with the State Street Consumer Staples Select S
- 📰 [Bots don't fly here: United Airlines restricts AI agents from booking flights](https://finnhub.io/api/news?id=29a0a78d30ac55ef7c4c72e36b1df6cb5fa147c2c7e634daa398f89ababd098d) — Yahoo：AI agents just got denied boarding at United Airlines.
- 📰 [Expedia and Booking shares rise after United Airlines blocks AI booking](https://finnhub.io/api/news?id=e3af5f930d0d110dcef644cd0555699a063ee777ddb1ce5ee332b9072cbc5ca1) — Yahoo：Investing.com -- Expedia Group Inc and Booking Holdings Inc shares gained Wednesday after United Airlines announced it will block personal artificial-intelligen…

**Xylem Inc.（XYL -1.79%）**
- 📰 [Xylem Inc. (XYL) Presents at Baird 2026 Global Industrial Conference 2026 Transcript](https://finnhub.io/api/news?id=ee90ce670804983bdf3fd6f10ace5aa6a7920dd622aaf195d790e2783df2dff5) — SeekingAlpha：Xylem Inc. (XYL) Baird 2026 Global Industrial Conference 2026 November 10, 2026 2:45 PM ESTCompany ParticipantsMatthew Pine - CEO, President &...
- 📰 [A look back at water infrastructure stocks’ Q3 earnings: Xylem (NYSE:XYL) vs the rest of the pack](https://finnhub.io/api/news?id=367b3b92e0f3ba4aa54de69538b25bc8ffe156ba7b33e3a9c8b007f53c45e9dc) — Yahoo：Earnings results often indicate what direction a company will take in the months ahead. With Q3 behind us, let’s have a look at Xylem (NYSE:XYL) and its peers.

**Tesla, Inc.（TSLA +2.91%）**
- 📰 [Nasdaq 100 Futures Drop As Reports Relating To Jensen Huang’s Ousting Outweigh CoreWeave’s Strong Outlook — NVDA, TSLA, SNDK, CAPR, CRWV In Focus](https://finnhub.io/api/news?id=d7e64c7d1c923e54529848fa5f0a9c68b37f6ea780e8006c16005e87309242b6) — Yahoo：Nvidia’s board is preparing to meet as soon as today to determine Jensen Huang’s future as the chipmaker’s chief executive, WSJ reported.
- 📰 [3 Growth Stocks With At Least 18% Revenue Growth](https://finnhub.io/api/news?id=b3f36348185de2ad346df2295f4177c2d3ef1fb2d68d1e1bc74f0b8199cc7926) — Yahoo：Anthropic’s planned IPO, with a potential valuation in the trillions and tens of billions of dollars on the table, shows how much capital is chasing high growth…
- 📰 [Tesla extends Cybertruck production to clear backlog, Musk says](https://finnhub.io/api/news?id=eec3b0c01ccfa6c1b0eb2ede12efd139d31884e586d233caf3fb26ea7e366864) — Yahoo：Tesla CEO Elon Musk originally said Tesla would wind down production​ at its Texas factory in mid-2027.

**Expedia Group（EXPE +2.37%）**
- 📊 Argus Research 維持：Buy，目標價 360 → 300
- 📊 Jefferies 維持：Buy，目標價 350 → 325
- 📰 [United Is the Latest to Block AI Agents Like Muse. These Stocks Have a Mini Relief Rally.](https://finnhub.io/api/news?id=6fdd00c104889e999f3a063943a86bb6b319882f95a981895521afd7101c621d) — Yahoo：Some of the hardest hit stocks after Muse’s release in September got a bounce on Wednesday.
- 📰 [Expedia and Booking shares rise after United Airlines blocks AI booking](https://finnhub.io/api/news?id=e3af5f930d0d110dcef644cd0555699a063ee777ddb1ce5ee332b9072cbc5ca1) — Yahoo：Investing.com -- Expedia Group Inc and Booking Holdings Inc shares gained Wednesday after United Airlines announced it will block personal artificial-intelligen…
- 📰 [Airbnb and Travel Stocks Fall as OpenAI Acquisition Raises New AI Threat](https://finnhub.io/api/news?id=4f4c845b9491b2289ad70a16bd45fe70d55eb54d5467c12fc4de12b37dd822bd) — Yahoo：OpenAI’s acquisition of Odessia raises fresh concerns about artificial-intelligence disruption.

**Chipotle Mexican Grill（CMG +2.00%）**
- 📰 [Can Chipotle Stock Hit $54?](https://finnhub.io/api/news?id=d980bc05c7ae13a7027958c8b23a6ff82853d78b6f16b12a6cefaacf264424f2) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [Can Chipotle's 1,700 Upgraded Kitchens Revive Traffic in 2027?](https://finnhub.io/api/news?id=2d381c5d67d0d51874c943ac0132ed4877ee7f489f7ef7ba392f3a5c38faed6e) — Yahoo：CMG's 1,700-plus upgraded kitchens are lifting sales and speeding service. Can the broader rollout revive traffic in 2027 despite weak consumer demand?
- 📰 [Should You Approach Dutch Bros Stock With Caution After Q3 Results?](https://finnhub.io/api/news?id=6e4332144c2cb88793096015a7b05b43442949a1123854c7c2766fdf1f393acb) — Yahoo：Should BROS' strong Q3 beat be overshadowed by slowing customer traffic, rising operating costs, aggressive expansion and a premium valuation?

**Airbnb（ABNB -3.09%）**
- 📊 Jefferies 維持：Buy，目標價 215 → 230
- 📊 Rothschild & Co 升評：Sell → Neutral，目標價 135 → 190
- 📰 [Airbnb vs. MercadoLibre: Which Consumer Stock Is a Better Buy for 2027?](https://finnhub.io/api/news?id=1cbb86f3d04039a43db54612b5394c0ad056534c79c2c9063209d9e48a4eb987) — Yahoo：Airbnb's lean marketplace model and MercadoLibre's sprawling Latin American ecosystem pull in very different directions, and the margin gap between them tells a…
- 📰 [Should You Invest in Airbnb (ABNB) Based on Bullish Wall Street Views?](https://finnhub.io/api/news?id=bef69a37e4191992813269ed41bdd07a7b09613d0ab1ecebf013124530f394c1) — Yahoo：According to the average brokerage recommendation (ABR), one should invest in Airbnb (ABNB). It is debatable whether this highly sought-after metric is effectiv…
- 📰 [Are Consumer Discretionary Stocks Lagging Airbnb (ABNB) This Year?](https://finnhub.io/api/news?id=b96cfaa29e15a2a211ef924f48e9bf25347b28d1bfc52843050b69cf58aa4a5f) — Yahoo：Here is how Airbnb, Inc. (ABNB) and Sony (SONY) have performed compared to their sector so far this year.

**Lululemon Athletica（LULU -2.65%）**
- 📊 Cantor Fitzgerald 首評：Neutral，目標價 121
- 📰 [Every December the Nasdaq-100 Reshuffles. QQQ Holders Feel It Even If They Never Trade](https://finnhub.io/api/news?id=6eb0188d7a0dc384645ddb4fbfc15e253548dcf3d024cf220c44891aefda8d82) — Yahoo：Every December, the Nasdaq-100 reshuffles. When the index changes its lineup, your fund's holdings change that week whether you do anything or not.
- 📰 [Michael Burry Re-Enters LULU On Weakness, Boosts Positions In MercadoLibre, PayPal, JD And More](https://finnhub.io/api/news?id=251c5a3c76ee601eea34d8209eb63d4b008fd10209e95366b072c64c28d78bc1) — Yahoo：Burry cited lululemon’s new management’s failure so far to reverse a prolonged “black hole” in marketing and messaging, which has left the stock overlooked duri…
- 📰 [Michael Burry buys back into Lululemon, adds to other positions](https://finnhub.io/api/news?id=6247ff85c85e26cecd39e8df29b372dc5d9b5d4637f3638b6920e84ed10e270e) — Yahoo：Investing.com -- Michael Burry disclosed that he purchased shares of Lululemon on Wednesday after previously exiting his position in the athletic apparel compan…

**McDonald's（MCD -2.61%）**
- 📰 [Scalare Partners Provides Corporate Update on Portfolio Company 10XU](https://finnhub.io/api/news?id=2dc4d4fb2c9e27c1985cccc02a649723a1f81e915eed101143b48f23180bd9af) — Yahoo：Sydney, Australia--(Newsfile Corp. - November 11, 2026) - Scalare Partners Holdings Limited (ASX: SCP) ("Scalare" or "the Company") today provides a corporate u…
- 📰 [Nvidia’s Board Is Reportedly Preparing to Oust Jensen Huang. When Intel and McDonald’s Pushed Out CEOs Over Workplace Relationships, Shareholders Got Opposite Results](https://finnhub.io/api/news?id=fffb4e76d93a0e08167eea3a0746446c12793b434698c221801c4f9ecd1ac8ad) — Yahoo：Nvidia's board is preparing to oust Jensen Huang, the Wall Street Journal reported, after a Times report of his relationship with executive Kimberly Powell; Nvi…
- 📰 [McDonald’s or Home Depot: I Can Only Pick One Dividend Stock for the Next 20 Years. Here’s Mine](https://finnhub.io/api/news?id=69ffa5e12d896549d9d4b93bcc79848cdce1ba8759728dd02822b3038620f7a5) — Yahoo：Key TakeawaysMcDonald’s and Home Depot both yield 2. 8%, so $10,000 in either pays $280 a year today.

**Equinix（EQIX +1.72%）**
- 📰 [Equinix (EQIX) Lands First Silicon Spin Quantum Computer And €2.5 Billion Green Bond](https://finnhub.io/api/news?id=94e85ea8801b2e86ea3ca1e9abc3d3649d357ad56d80747b8cba6941b9e8b21b) — Yahoo：Equinix (NasdaqGS:EQIX) has hosted Diraq's world-first silicon spin quantum computer inside one of its data centers, marking an early commercial deployment alon…
- 📰 [This data center stock remains a Top Pick, Deutsche Bank says](https://finnhub.io/api/news?id=92ad245d859d40e74ece465524ccc9224c8a8611cf0beffff9c86d153d808ad2) — Yahoo：Investing.com -- Deutsche Bank analysts believe the recent post-election sell-off in data center stocks has been overdone, with Equinix emerging as the firm’s t…
- 📰 [Iron Mountain Raises Its Dividend 15% as Data Center Revenue Surges 34%](https://finnhub.io/api/news?id=0672138d8b8bd93b9765f33011d03730bff44feb109bda672469bcec1da2e6d1) — Yahoo：A company famous for storing paper boxes now funds its shareholder payouts with server halls. Income investors should weigh how Iron Mountain stacks up against…

**Iron Mountain（IRM +1.49%）**
- 📰 [Iron Mountain Raises Its Dividend 15% as Data Center Revenue Surges 34%](https://finnhub.io/api/news?id=0672138d8b8bd93b9765f33011d03730bff44feb109bda672469bcec1da2e6d1) — Yahoo：A company famous for storing paper boxes now funds its shareholder payouts with server halls. Income investors should weigh how Iron Mountain stacks up against…
- 📰 [Winners Of REIT Earnings Season](https://finnhub.io/api/news?id=e10b8b7663d556f0d6e44c5b98dc7f02402b9cd11ec56a074d87030e6cc23d95) — SeekingAlpha：REITs deliver one of their strongest earnings seasons on record: 78 (80%) raise FFO guidance while outperforming the S&P 500 for five straight weeks.
- 📰 [Iron Mountain to Present at Upcoming Financial Conference](https://finnhub.io/api/news?id=6c7daa2e16bc58bf092247a36ea49f8df4ec88808750980675521da3f1cac040) — Yahoo：PORTSMOUTH, N.H., November 10, 2026--Iron Mountain Incorporated (NYSE: IRM), a global leader in information management services, announced today that it will pr…

**Alexandria Real Estate Equities（ARE +0.85%）**
- 📰 [Winners Of REIT Earnings Season](https://finnhub.io/api/news?id=e10b8b7663d556f0d6e44c5b98dc7f02402b9cd11ec56a074d87030e6cc23d95) — SeekingAlpha：REITs deliver one of their strongest earnings seasons on record: 78 (80%) raise FFO guidance while outperforming the S&P 500 for five straight weeks.
- 📰 [Alexandria Real Estate: 3 Key Takeaways for Long-Term Investors](https://finnhub.io/api/news?id=f59c0e293d4378e51275e6f27b497b518775844bf3cb1a0ce59930541ef88fd3) — Yahoo：Tech demand is reshaping the tenant mix at Alexandria's research properties, widening its customer base beyond biotech.

**Vivmark Residential（VMRK -1.70%）**
- 📊 UBS 維持：Buy，目標價 71 → 74
- 📊 Cantor Fitzgerald 維持：Neutral，目標價 70 → 68
- 📰 [Renters Are Staying Put. These 4 Apartment REITs Could Benefit](https://finnhub.io/api/news?id=c7455755f1c1e492d84d6d84f09e3d3206ca6148b18e120ebe397edf8b621d74) — Yahoo：Fewer renters are buying homes, and apartment landlords are quietly collecting the payoff. We checked four REIT dividends against their cash flow, and the one p…
- 📰 [Vivmark Residential Calls Series K Cumulative Redeemable Preferred Shares for Redemption](https://finnhub.io/api/news?id=bb03702b514cb9ff21a72334b1eea2dc1b3fa4a5473c2fddbe336a85eaed4d7d) — Yahoo：ARLINGTON, Va. & CHICAGO, November 10, 2026--Vivmark Residential (NYSE: VMRK) (the "Company") today announced that it has called for redemption, on December 10,…
- 📰 [UBS Maintains Buy on Vivmark Residential, Raises Price Target to $74](https://finnhub.io/api/news?id=551f7d798bb83021fecea72eec4cad44399a81d641c041a28081331ebd560475) — Benzinga：UBS analyst Ami Probandt maintains Vivmark Residential (NYSE:VMRK) with a Buy and raises the price target from $71 to $74.

**Camden Property Trust（CPT -0.80%）**
- 📊 Morgan Stanley 維持：Equal-Weight，目標價 118 → 107
- 📊 UBS 維持：Neutral，目標價 101 → 107
- 📊 Cantor Fitzgerald 維持：Neutral，目標價 111 → 101
- 📰 [Renters Are Staying Put. These 4 Apartment REITs Could Benefit](https://finnhub.io/api/news?id=c7455755f1c1e492d84d6d84f09e3d3206ca6148b18e120ebe397edf8b621d74) — Yahoo：Fewer renters are buying homes, and apartment landlords are quietly collecting the payoff. We checked four REIT dividends against their cash flow, and the one p…
- 📰 [UBS Maintains Neutral on Camden Prop Trust, Raises Price Target to $107](https://finnhub.io/api/news?id=6279daea4608db7fda1c1920fcd9de74fcae6120c162bb8503d96acd33305508) — Benzinga：UBS analyst Ami Probandt maintains Camden Prop Trust (NYSE:CPT) with a Neutral and raises the price target from $101 to $107.

**Invitation Homes（INVH -0.78%）**
- 📰 [Invitation Homes Announces Addition of M. Colin Connolly to Board of Directors](https://finnhub.io/api/news?id=95618a4f9accf010f5a2910ab15551c875460137703ad89baecd6489b7bfb663) — Yahoo：DALLAS, November 11, 2026--Invitation Homes Inc. (NYSE: INVH) ("Invitation Homes" or the "Company"), the nation’s premier single-family home leasing and managem…
- 📰 [Winners Of REIT Earnings Season](https://finnhub.io/api/news?id=e10b8b7663d556f0d6e44c5b98dc7f02402b9cd11ec56a074d87030e6cc23d95) — SeekingAlpha：REITs deliver one of their strongest earnings seasons on record: 78 (80%) raise FFO guidance while outperforming the S&P 500 for five straight weeks.
- 📰 [Vonovia: Long-Term Modelling Shows Resilience; A Short-Term Positive Trigger Is Probable](https://finnhub.io/api/news?id=bbe27ab70745a8b935de2f0a9f76821c8cec86d49a7a057d4eeebc34aa848977) — SeekingAlpha：Vonovia SE trades at a deep discount after a >40% share price drop, driven by rate expectations and Berlin's political risk. See why VONOY stock is a Buy now.

**Regeneron Pharmaceuticals（REGN +4.73%）**
- 📰 [What Will Decide Vertex Stock Over The Next Six Months?](https://finnhub.io/api/news?id=23e74cf56d698e770dc5edd9f680d7ae95d023a8d86ade2823c5b853838b98f3) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [S&P 500 Premarket Movers: Gainers and Losers Ahead of the Open](https://finnhub.io/api/news?id=0218f22724d9afec1d16c9c7b82b38e00deb15f27da4d17f2a66eeda38bb8166) — ChartMill：S&P 500 premarket movers: COHR, LITE, STX, MRVL and WDC lead gainers, while ACGL, CLX, ABNB and REGN slip ahead of the opening bell.
- 📰 [iShares Biotechnology ETF (IBB) Adds Gilead and Amgen on Nov. 9](https://finnhub.io/api/news?id=c8f93063dc31c5557e8cbe55e9d2f4ec901c36246ff5e3a926a6ba698fb49f0d) — Yahoo：The $11.34 billion fund bought $230.1 million of stock on Monday, Nov. 9, lifting 231 positions and trimming none.

**DaVita（DVA +2.35%）**
- 📰 [What Happened To People Who Bought Quest Stock At A High?](https://finnhub.io/api/news?id=e0fa8e3baf82cfbb33f2a84d91ae88e8e2d07f0fa9238c1dd10a6fb5e97f028b) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [DaVita Inc. (DVA) Presents at 8th Annual Wolfe Research Healthcare Conference Transcript](https://finnhub.io/api/news?id=01dc74d765b0cb32a8a7d0571a6ced767af338375c47adf0409a422f1f89e80f) — SeekingAlpha：DaVita Inc. (DVA) 8th Annual Wolfe Research Healthcare Conference November 10, 2026 8:40 AM ESTCompany ParticipantsJoel Ackerman - CFO &...
- 📰 [DaVita Sees Growth and Drug Savings Offsetting 2027 Reimbursement Pressure](https://finnhub.io/api/news?id=34eb0ab95399ea87a2271ccb9f98b11f3ddcb5734db36b9d3428f026005f5e44) — Yahoo：DaVita (NYSE:DVA) expects stronger dialysis treatment growth and pharmaceutical cost savings to help offset reimbursement pressures in 2027, though Chief Financ…

**ResMed（RMD +1.89%）**
- 📰 [Oura makes bet on AI in healthcare with $625M Counsel Health buy](https://finnhub.io/api/news?id=73c537bf101f01627de9744360f2aaa2173bc718d8c574d6aff03fc2cc46499f) — Yahoo：The planned acquisition comes as wearable firms push to up their healthcare offerings.
- 📰 [INSP Is Up 12.1% in Three Months: Can Its Rally Continue From Here?](https://finnhub.io/api/news?id=8722160def3d2b22f5655ce3013bea82c4f0462b71a7aa6df3feb454fc62392d) — Yahoo：Inspire Medical shares have rallied 12.1% in three months, but reimbursement hurdles and weak revenues raise questions about whether the gains can continue.

**Stryker Corporation（SYK -1.70%）**
- 📊 Piper Sandler 維持：Overweight，目標價 325 → 350
- 📰 [Is Current Price Of Stryker Stock A Strong Support?](https://finnhub.io/api/news?id=68b0aee21a86199e38276207ddbb1621137a3b81914d056caad961c785cd4893) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [Ronda E Stryker Executes Sell Order: Offloads $178.92M In Stryker Stock](https://finnhub.io/api/news?id=73d4d377fde15befe9eaf20a013ea0a3fceb0d1c3738703b7e6b658f75ee0e72) — Benzinga：On November 9, a recent SEC filing unveiled that Ronda E Stryker, Director at Stryker (NYSE:SYK) made an insider sell. What Happened: A Form 4 filing from the U…
- 📰 [Stryker Corporation Increases Board Size to 12 and Elects James E. Davis](https://finnhub.io/api/news?id=d2c4d7ec166b374a2cac5e9176faa5927a7930eabf6acacd1121b859c65ed31d) — Fintel

**Solventum（SOLV -1.62%）**
- 📰 [Tracking Christopher Davis' Davis Selected Advisers 13F Portfolio - Q3 2026 Update](https://finnhub.io/api/news?id=4123f11d3d5c3d26224857fac9ae0233b8648eee6f0ac5c1f62aa7302b8fcfdd) — SeekingAlpha
- 📰 [SOLV Q3 2026 deep dive: ERP transition, separation milestones, and margin focus define quarter](https://finnhub.io/api/news?id=ec8665426a77e62308c65e4f0c85da5df0b1fcfbc897a76a4b4d590a01882f37) — Yahoo：Healthcare solutions provider Solventum (NYSE:SOLV) reported Q3 2026 results topping the market’s revenue expectations, but sales fell by 7% year on year to $1.…
- 📰 [Solventum (SOLV) stock trades up, here is why](https://finnhub.io/api/news?id=405c311240a71a4ec20066b6a444b951f7a436079a472745dd8a2a25afdc5b6a) — Yahoo：Shares of healthcare solutions provider Solventum (NYSE:SOLV) jumped 5% in the afternoon session after the company reported third-quarter 2026 financial results…

**GE HealthCare（GEHC -1.26%）**
- 📊 Argus Research 維持：Buy，目標價 85 → 110
- 📰 [GE HealthCare: 3 Key Takeaways for Long-Term Investors](https://finnhub.io/api/news?id=9be243cb5bf7ca67e4314143d94c56520c957f2bcc065f963ee6a0d62c07ba3e) — Yahoo：Multiyear supply contracts and higher-margin product launches may help position the company to sustain margin expansion amid industry chip shortages.
- 📰 [ALC's Q3 Earnings Beat, Revenues Miss, '26 Outlook Narrows, Stock Down](https://finnhub.io/api/news?id=1fb8db014faf0370433d03d2354152ba2953646ef38fa8b85491c0ce0c734ee7) — Yahoo：Alcon's Q3 earnings top estimates as sales miss, margins expand and 2026 growth targets narrow; shares slip after the report.
- 📰 [Moat Stocks Rebound As Tech Retreats](https://finnhub.io/api/news?id=b8e75a5f7e03cbcab16edf1449365f2ccb2392f260f9f625c7df68094140d699) — SeekingAlpha


**當日財報｜2026-11-11（S&P 500 成分股，依營收規模排序）**

- 當日無 S&P 500 成分股公布財報

## 6. Treasury Yield Curve

In [6]:
def treasury_curve():
    frames = []
    for yr in (AS_OF.year, AS_OF.year - 1):          # 年初資料不足兩天時補前一年
        url = ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/"
               f"{yr}/all?type=daily_treasury_yield_curve&field_tdr_date_value={yr}&page&_format=csv")
        r = HTTP.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        frames.append(pd.read_csv(io.StringIO(r.text)))
        if sum(len(f) for f in frames) >= 2:
            break
    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    return df.sort_values("Date", ascending=False).reset_index(drop=True)


def curve_shape(d2, d10):
    """依 2 年、10 年殖利率變動（bp）判斷曲線型態"""
    if abs(d10 - d2) < 1:
        return "大致平行移動（2 年與 10 年變動相近）"
    if d2 <= 0 and d10 <= 0:
        return "牛平（長端降更多）" if d10 < d2 else "牛陡（短端降更多）"
    if d2 >= 0 and d10 >= 0:
        return "熊陡（長端升更多）" if d10 > d2 else "熊平（短端升更多）"
    return "扭轉變平（短升長降）" if d2 > 0 else "扭轉變陡（短降長升）"


def fred_curve():
    """備援：FRED 的公債殖利率（資料同樣來自財政部），5 個天期同時抓"""
    if not FRED_API_KEY:
        raise ValueError("未設定 FRED_API_KEY")
    cols = {"3 Mo": "DGS3MO", "2 Yr": "DGS2", "5 Yr": "DGS5", "10 Yr": "DGS10", "30 Yr": "DGS30"}

    def series(sid):
        try:
            r = HTTP.get("https://api.stlouisfed.org/fred/series/observations", timeout=TIMEOUT,
                         params={"series_id": sid, "api_key": FRED_API_KEY, "file_type": "json",
                                 "observation_start": f"{AS_OF - timedelta(days=30):%Y-%m-%d}"})
            r.raise_for_status()
        except requests.RequestException as e:
            raise RuntimeError(safe_err(e)) from None      # FRED 的金鑰只能放在網址，錯誤訊息要先遮蔽
        return pd.to_numeric(pd.Series({o["date"]: o["value"] for o in r.json()["observations"]}), errors="coerce")

    with ThreadPoolExecutor(len(cols)) as ex:
        df = pd.DataFrame(dict(zip(cols, ex.map(series, cols.values())))).dropna()
    df.index = pd.to_datetime(df.index)
    return df.rename_axis("Date").reset_index().sort_values("Date", ascending=False).reset_index(drop=True)


try:
    tc, source = treasury_curve(), "美國財政部"
except Exception as e:
    print(f"財政部資料無法取得（{safe_err(e)}），改用 FRED")
    tc, source = fred_curve(), "FRED"
cur, prev = tc.iloc[0], tc.iloc[1]
rows = ["| 天期 | 殖利率 | 變動 |", "|:---|---:|---:|"]
for t in ["3 Mo", "2 Yr", "5 Yr", "10 Yr", "30 Yr"]:
    rows.append(f"| {t} | {cur[t]:.2f}% | {(cur[t] - prev[t]) * 100:+.0f} bps |")
d2, d10 = (cur["2 Yr"] - prev["2 Yr"]) * 100, (cur["10 Yr"] - prev["10 Yr"]) * 100
spread, spread_prev = (cur["10 Yr"] - cur["2 Yr"]) * 100, (prev["10 Yr"] - prev["2 Yr"]) * 100
stale = "" if cur["Date"].date() == AS_OF.date() else (
    f"\n\n⚠️ 這是 {cur['Date']:%m/%d} 的曲線，不是報告日 {AS_OF:%m/%d}：財政部約在美東 15:30–18:00 公布當日數字，"
    "債市休市日（如退伍軍人節、哥倫布日）不更新。稍後重跑本格；當日 10 年期變動先看第 3 格的美債。")
display(Markdown(f"**美債殖利率｜{cur['Date']:%Y-%m-%d}（vs {prev['Date']:%m/%d}，{source}）**\n\n" + "\n".join(rows) +
                 f"\n\n- 2s10s 利差：{spread:+.0f} bps（前一日 {spread_prev:+.0f} bps）\n- 曲線型態：{curve_shape(d2, d10)}" + stale))

**美債殖利率｜2026-11-10（vs 11/09，美國財政部）**

| 天期 | 殖利率 | 變動 |
|:---|---:|---:|
| 3 Mo | 4.30% | +6 bps |
| 2 Yr | 4.73% | +17 bps |
| 5 Yr | 4.81% | +14 bps |
| 10 Yr | 4.98% | +10 bps |
| 30 Yr | 5.35% | +9 bps |

- 2s10s 利差：+25 bps（前一日 +32 bps）
- 曲線型態：熊平（短端升更多）

⚠️ 這是 11/10 的曲線，不是報告日 11/11：財政部約在美東 15:30–18:00 公布當日數字，債市休市日（如退伍軍人節、哥倫布日）不更新。稍後重跑本格；當日 10 年期變動先看第 3 格的美債。

## 7. Free news

In [7]:
CNBC = "https://www.cnbc.com/id/{}/device/rss/rss.html"


def google_news(query):
    return ("https://news.google.com/rss/search?q=" + quote_plus(f"{query} when:1d")
            + "&hl=en-US&gl=US&ceid=US:en")


GROUPS = {   # (來源名稱, 網址)
    "盤勢與個股": [("CNBC", CNBC.format(100003114)), ("CNBC Earnings", CNBC.format(15839135)),
                   ("CNBC Investing", CNBC.format(15839069)), ("Google News", google_news('"stock market today"')),
                   ("Finnhub", "finnhub:general")],
    "總經與聯準會": [("CNBC Economy", CNBC.format(20910258)),
                     ("Fed 新聞稿", "https://www.federalreserve.gov/feeds/press_all.xml"),
                     ("Fed 演講", "https://www.federalreserve.gov/feeds/speeches.xml")],
    "美債與美元": [("Google News", google_news('"Treasury yields"')), ("Google News", google_news("dollar forex"))],
    "黃金": [("Google News", google_news('"gold prices"'))],
    "原油": [("CNBC Energy", CNBC.format(19836768)), ("OilPrice", "https://oilprice.com/rss/main"),
             ("Google News", google_news('"oil prices"'))],
    "中文（鉅亨網美股）": [("鉅亨網", "cnyes:us_stock")],
}


def fetch_items(name, url):
    if url == "finnhub:general":
        if not FINNHUB_API_KEY:
            return []
        return [(datetime.fromtimestamp(d["datetime"], timezone.utc), clean(d.get("headline"), 300), d.get("url", ""),
                 f"Finnhub / {d.get('source', '')}", clean(d.get("summary")))
                for d in finnhub("news", category="general")
                if d.get("datetime") and not blocked(d.get("url"), d.get("source"))]
    if url.startswith("cnyes:"):
        r = HTTP.get(f"https://api.cnyes.com/media/api/v1/newslist/category/{url[6:]}",
                     params={"limit": 50}, timeout=TIMEOUT)
        r.raise_for_status()
        return [(datetime.fromtimestamp(d["publishAt"], timezone.utc), clean(d.get("title")),
                 f"https://news.cnyes.com/news/id/{d['newsId']}", name, clean(d.get("summary")))
                for d in r.json()["items"]["data"]]
    r = HTTP.get(url, timeout=TIMEOUT)
    r.raise_for_status()
    items = []
    for e in feedparser.parse(r.content).entries:
        t = e.get("published_parsed") or e.get("updated_parsed")
        src = e.get("source") or {}
        if src and blocked(src.get("href"), src.get("title")):
            continue                                   # 略過 Reuters、Bloomberg 與付費牆媒體
        title, publisher = clean(e.get("title"), 300), src.get("title", "")
        if publisher and title.endswith(" - " + publisher):
            title = title[: -(len(publisher) + 3)]
        items.append((datetime(*t[:6], tzinfo=timezone.utc) if t else None, title, e.get("link", ""),
                      f"{name} / {publisher}" if publisher else name, "" if publisher else clean(e.get("summary"))))
    return items


def merge(fetched):
    """同一組各來源依序合併：只留最近 NEWS_HOURS 小時、標題去重、新到舊取前 NEWS_PER_GROUP 則"""
    cutoff = datetime.now(timezone.utc) - timedelta(hours=NEWS_HOURS)
    seen, items, errs = set(), [], []
    for name, (its, err) in fetched:
        if err:
            errs.append(f"{name}：{err}")
        for it in its or []:
            key = re.sub(r"\W+", "", it[1].lower())
            if it[0] and it[0] >= cutoff and it[1] and key not in seen:
                seen.add(key)
                items.append(it)
    items.sort(key=lambda x: x[0], reverse=True)
    return items[:NEWS_PER_GROUP], errs


# 所有來源一次同時抓（原本是每組內逐一抓），再依組別合併
tasks = {(g, i): ("news", partial(fetch_items, name, url))
         for g, sources in GROUPS.items() for i, (name, url) in enumerate(sources)}
res = gather(tasks, {"news": len(tasks)})
results = {g: merge([(name, res[g, i]) for i, (name, _) in enumerate(sources)]) for g, sources in GROUPS.items()}

lines = []
for g, (items, errs) in results.items():
    lines += [f"### {g}"]
    for when, title, link, src, summary in items:
        title = title.replace("[", "(").replace("]", ")")
        lines.append(f"- `{when.astimezone(TW):%m/%d %H:%M}` [{title}]({link}) — *{src}*" + (f"：{summary}" if summary else ""))
    if not items:
        lines.append("- （無）")
    if errs:
        lines.append(f"- ⚠️ 無法取得：{'、'.join(errs)}")
    lines.append("")
display(Markdown("\n".join(lines)))

### 盤勢與個股
- `11/12 09:14` [Cramer's stance on Nvidia's stock amid reports that CEO Jensen Huang may be ousted](https://www.cnbc.com/investingclub/2026/11/11/jim-cramer-jensen-huang-nvidia-stock-advice.html) — *CNBC*：Jim Cramer said Wednesday night that he was saddened by a news report that indicated Nvidia's board of directors is preparing to fire Huang.
- `11/12 08:55` [Why B&G Foods Stock Soared Today](https://news.google.com/rss/articles/CBMijAFBVV95cUxObzlubXRhWHIzcG5jNjBNMWpsTXNpMUxNUXpRNl91RGxtUmJ1eHp2OXVMNHZBNTFla2h2bmViZjc0U2xuN2ZPZEVhMmVmc2pYSkZza3FENDl0RkR6d3FNaWI0emhOUVJONFUyR3I3VHlSUnRjbFVjRVRnWWFiaFVsTXY1NW5nUHZYaWVwQQ?oc=5) — *Google News / Yahoo Finance*
- `11/12 08:46` [Meta head of infrastructure will leave and be succeeded by former AWS cloud exec](https://www.cnbc.com/2026/11/11/meta-infrastructure-departure-aws-cloud.html) — *CNBC*：Meta's head of infrastructure and co-head of engineering Santosh Janardhan will leave in 2027, with former AWS senior vice president David Brown taking over.
- `11/12 08:43` [Why Anterix Stock Climbed Higher Today](https://news.google.com/rss/articles/CBMingFBVV95cUxPQzBfc1NqdVpZX3UzTGwtcWgzWnlaZWdNZFFMR1VsdmVvUG5QSzk4UXdReWtaYlRxbExnSVhDNnZQem9IaTZ6dlBIOXQ2cFhUNHVPZzlVb0oyMUlxcFVRcmUydjlOQlhaVWdTd3ZIWmVPTkRiYUJ3NFR2ZWNUZlhGeE1zaGtIbE9VNWpFTXc3QkV1WkdJcEw0VFRLYzhBUQ?oc=5) — *Google News / Yahoo Finance*
- `11/12 08:33` [Stock Market Today: Nvidia Stock Slides 8% After Hours as Board Weighs Jensen Huang’s Future](https://news.google.com/rss/articles/CBMizgFBVV95cUxOTEdzSDBUcDRkbjFSR2RCYVNiR2c1NUUwRnNibUxEOGhNOVdPVkltT2VzcVhzeVFuZFAzZndUWE5RYzdQdXg5N1VCdXdDRDJBMWhJRlAtWXg3NTNteFB5YnpLa2ZuWDJ5ODRFR2h1SWI1UUlkVDZvVE4tSE9SYzVxVEZuZ2ZiaUV3b2VxQmpfVXptRWpKWktOV0hFMDBjc2lEMjVJbzMyQk8zT1AyR3lsVDVLTUhhcE1hRE1JMlZKYkNTZVZaeDJ4TXYyU3REdw?oc=5) — *Google News / Eurasia Business News*
- `11/12 08:21` [Nvidia prepares to oust Jensen Huang, WSJ reports — stock sinks](https://www.cnbc.com/2026/11/11/nvidia-jensen-huang-affair-employee.html) — *CNBC*：The report comes days after Huang admitted to an affair with a woman who is believed by U.S. intelligence of being a Chinese spy.
- `11/12 07:57` [Inside India newsletter: Trump's poll setback fuels hope for U.S.-India trade deal by year end](https://www.cnbc.com/2026/11/12/us-india-trade-tariffs-trump-modi.html) — *CNBC*：While American voters are more affected by import duties on Canadian goods, experts said that a trade deal with India could blunt criticism against tariffs.
- `11/12 07:42` [Nvidia's stock dives as board weighs move to oust Jensen Huang](https://news.google.com/rss/articles/CBMingFBVV95cUxOM3V6b0VlNWlnYlE5aTlJNUVKeU9WUXQyVFBxUGIwYldCd3EzczdrbFdXazVxUi1KblpZM1NTaFlmM29WWl9yV0ZSaWdGcHlVWGdyMmlnVVR4WEZVQ2pKRldibDlrTWhWVGZwbEppeEJfbmNxOENGOU00dXJVQU1xSFdGMlp2V21LYXJEZzhPRG5JeF9lZjVkbkxWMG1wQQ?oc=5) — *Google News / Yahoo Finance*

### 總經與聯準會
- `11/12 06:59` [Analysis: Is Nvidia too big to fail? Its AI financing web raises the stakes](https://www.cnbc.com/2026/11/11/nvidia-jensen-huang-ai-financial-system-risk.html) — *CNBC Economy*：The tech giant's web of ties to other companies is raising the possibility that it could do far-reaching economic and financial damage.
- `11/11 20:27` [Consumers are showing signs of strain even as they keep spending, CNBC/NRF Retail Monitor shows](https://www.cnbc.com/2026/11/11/october-retail-sales-cnbc-nrf-2026.html) — *CNBC Economy*：Retail sales rose at their slowest pace in nearly a year in October as American consumers showed signs of strain from high prices.
- ⚠️ 無法取得：Fed 新聞稿：HTTPError: 404 Client Error: Not Found for url: https://www.federalreserve.gov/feeds/press_all.xml

### 美債與美元
- `11/12 09:26` [Rupee climbs to near 7-month high of 93.69 against dollar](https://news.google.com/rss/articles/CBMi0gFBVV95cUxPQmJiamtuSXkzMlZjeWNtd28xX2xZOTRHOUREbTNRSmkwRldxVU9CSnpia1dJNTVUSjhCOWVqMTZOVlkya0VJUVBVSjFtMzBLT0FnMUNuVENVZjRlcnR4SmJLTHVPQnpvR3dnZnBLUktldkJLRWlkdzlvSkdEbFZIRmtURGhELTYtdHRaek1VNG80dm5iNzQ3WldWTlpFR1VneW5mNm8yYTFYQkozZTIzYVVLMEFySGE5Q0Z0RlE2ZGFHaGZXMG1VbXExeGtWWFVsdHfSAdcBQVVfeXFMTlZOOEVzNl9qcnp2SUFmdzl3UjZUeDVOeWpCSF9EaENQdUtMLXBmVUs0NWpMYTVlT0ZobFdpaVFWYkJldU5LOEFDalRDMnpkYnlFbGZ1Q2o5X05tR3ViZl9SQUtyNDcwTEZoMnU3S0VBS1E4aHIzNURza2d5S0xWMThpVUw1U0x2TWdhdGlSU2F1NUF4SDZvbFRrYl95eVVrVzZxVWp3Z0wwZUNTRkh5VUk3b29DYnJlZE1oakZNdXpPd0FEbGJhamtSOS1JYi1NVEIxeC05c0U?oc=5) — *Google News / The Economic Times*
- `11/12 09:06` [Ringgit opens almost flat against US dollar amid cautious sentiment](https://news.google.com/rss/articles/CBMixwFBVV95cUxQSk9DYUFqVDNWdUdiaTBFcDNqWmZ1ODc5OS1YdHU2ay0xWlJJc3NiNzUxRU43X1NROEdQUUpncWpmRmZMU1h0WlUyQXJkWWhzSVBxcFZEQ2R3WG5BUW1oY1VpTTlIQk9zU0tMUnhrQ0NNbmxiaF9TakMxUUtDbWkwbkduSVFDRWdsREFEUThPZmoyXzZmR0dDaUQwZTBvRzRBc2VXLTY5SUt6V3FCVVRxNk9udDhkVEJiZFByVFhuMEJzZ1FQQUs0?oc=5) — *Google News / The Star*
- `11/12 08:54` [US Dollar Forecast: AUD/USD, Gold, EUR/USD Under Pressure](https://news.google.com/rss/articles/CBMinwFBVV95cUxNZkt5UzVVeVZSd3p1NjZCdXJsQUJGeEFfMjFiYlUzckROQTZMS2VDV2NqdVdxRi1fcEhKdW9id1VDcWQtQWktZjhfT0NLT3Fqdy1HeXpUR3E0VGZtZmNKelAzS0NHaC1NVnRQLVJJQXF3ZFU3NTR2N3o2T1NkMExxSGJLY3ZneE0tamNXQWFrMjhPOEJwOVVvQUFtVnRJWDQ?oc=5) — *Google News / FOREX.com*
- `11/12 08:49` [Dollar near one-month high as yen edges toward 160](https://news.google.com/rss/articles/CBMiyAFBVV95cUxNX2Y5cklmWXVsdUhZc1ZsU0JfVHJ3ZW9QTzVjSEl5aFowdl9ncHJNcnRoU19RVkNCSmg4cG9HM1d6NHlwcm5BaTA0X3d5SnNRMzcyMllWSVFYX3Rsd2FzY1VyRlJqZTFBX1Jac183YkJTaFpId2t3bmhYU05LQnhRbGhTR3djQUFzejJmMTNyU0JkN1Y1d3NkTHFHYUFVMFFQQk91TW1pSS1VcGdFbVRSdHExVDBwdmZEbTNQVjR1VGNDUmNEdzFPaNIBzgFBVV95cUxQU202LWxBRUFrV004aV83c3FISHZXR0MyQ0RxbFZoc0Y2N3VVdVRqRzY4THVOc1NfRGhQa3NIQkw1R25rTFpCcTFfWVJPY2pHWW03R0NyZGp5bzZDblZ0bVJCb3ZVSkVkdjRhUzNTQS1qWGMzb3VmdGoxVW9uRUF5UGVyNUh4NGFCZmE5T3l1SVFCbWtxc2NuSzhtRU91RlJoTlc4RnBIZ3huQ2Qta3F4VG9oRE4xaUQzcFF6T1QxaG54TVdkM1o0RHgwYVpnUQ?oc=5) — *Google News / The Economic Times*
- `11/12 08:46` [Gold (XAUUSD), Silver, Platinum Forecasts – Gold Drops As Dollar Tests Yearly Highs](https://news.google.com/rss/articles/CBMixwFBVV95cUxPZktBMEVrTlY5Q1ZUV3JtTDJBX3BUWjE2c0tJdGQxZ0JGc3BKb1dtc1F4Y1UySEotNWZ5ak5uUjdDUHlQRVN0bUJvLVAtVGw2dFFkWkkxUE9NWG1MNXVFaU12Rk8xam1XWHVrSXI3d3BWMC1NNDE5LWNoUWQ2T2lvMHRYeFRMTGthaldnVGQzZTR6UkdFQ21TUFl4bnNwUHN4aC1lbmdROFRjXzNVdlBrcWFmZ1hwMTNFUTVuM2tKWDlwRUlvYnZB?oc=5) — *Google News / FXEmpire*
- `11/12 08:33` [US Dollar Forecast: AUD/USD, Gold, EUR/USD Under Pressu](https://news.google.com/rss/articles/CBMinwFBVV95cUxQaXJBSUduY1BiZHpadnd3ZjNfRW9obUdfZm5IYm9nQ1c1cmp2elFtdHNSVVRYdlg2RGU2eG4wckEyZHdZMXF2Y1JXRFFzWmRnbElSNkVlV1F4VWtKSlF4VVQ4VWhiLTdmU2xyejN3OEtHcFp4QlZ0b1VYUTN3YWhWZ1U2NXlMaG9IV2NfUzRZQ0w3V0hnbkgwNHRITS1rd2M?oc=5) — *Google News / FOREX.com*
- `11/12 08:15` [Gold Price Forecast: Tests Support as Dollar Climbs](https://news.google.com/rss/articles/CBMiowFBVV95cUxQM1RuTnZFNUFEQWN1VU9TeTA3LWZGUGZZdDRoRE5LSkVJTGJ3Yi0zemhIVzUyekkyVDhRTWxMRUtLeGh1enNyUlJvc1FvaDJBdTRZVUQ1R0dpdERYM2JTN2thZW1GVElSS1ExYjdHRnZ4TlZIeWlhSDVhcUpCT2JuUFdHd19RLVJlT0VaYUVuX2xIcTlicHBPcHZzbUJ4WEhvOFFJ?oc=5) — *Google News / FXEmpire*
- `11/12 08:11` [News | Higher Treasury yields give buyers more bargaining power](https://news.google.com/rss/articles/CBMingFBVV95cUxQNnAwT0pMZ3BUVUptcW51dTktdXNBdUpCX0g2TnRMZ0hpYkJ1aVJZQlZVYWhYWEpvcnBxblJvUU9HTVN0Z0I4LXkwZUVUa3l0Tm9BLU41d1VzZE1zcXZhOEZaZV8tSWxibmJFLXRhOV9uMERGS0diT19jN011ck9feGJ0eGxvSVBoOUwxbnlWV2FBQmoxVUlfd3ktaC1lZw?oc=5) — *Google News / CoStar*

### 黃金
- `11/12 08:56` [Gold prices today, November 12th: Falling for the second consecutive session, dropping below the $4,000 mark.](https://news.google.com/rss/articles/CBMimwFBVV95cUxQbG5xTjVRS1ZiNWNhX0hzTGQwa2NUdzFDcDNqYmJ3RGdkSFVmelN2cG0yNWF1ZnNpeHpDdlY1ZnI5TXExeUt3S1dsdExrbmVSTFcxb002M25JcHNDRDcyN0xZcnlHS3BPR3o2UXFycW8zaUVfSWJzNFMtZndhSGNwaVdDRi1HMEhmQjRDR0ZKamtfc1Z0MzA5T2Jyaw?oc=5) — *Google News / Vietnam.vn*
- `11/12 08:17` [Gold falls as hot inflation figures, higher dollar weigh](https://news.google.com/rss/articles/CBMiqAFBVV95cUxNYVF6Uk1xa0k3eGdYdnJsWUF3eUJXek1pQUlrM2xpNmxfMUNHb2Q0S3pnV1hQdU84NGlwN0NGNjg0WGk4VGdaYkcxMzNmUmlPQnNkaWppX0NPQTJ4NEs3T2RHajdfTTB3MURGU3hUSk0wTTVtYjN5dHhwOE52T05jQ2gxcmU5dG1haHkyQndKUVZ1ZEoxbUJjSXZ5dFhzeFRrN3czTjBSSWk?oc=5) — *Google News / KLSE Screener*
- `11/12 06:30` [Gold price update, morning of November 12, 2026: SJC, 9999 DOJI rings, BTMC PNJ, Phu Quy.](https://news.google.com/rss/articles/CBMimwFBVV95cUxNNFQ3MlFMLVN3bmZtQTVlQkhjWDVOX1FZUmhJMmZOMVBqS3N0RnZETzdLYmYtaUZ3cnByY2ptZlgtVGpuY2swTDR0cUFodVJBSzFGQjBTeUlXY1RyV0pvdnlLYWhTQmc5OHlKYzNEeDZqV3UyT2QtNG10elBaS0FsMUN6akxiVXVQTXVYN21udmEtc3lHcWlxRWo4aw?oc=5) — *Google News / Vietnam.vn*
- `11/12 06:18` [Peru's mining exports hit $57.5 billion in eight months, with copper and gold driving a 55% year-over-year surge](https://news.google.com/rss/articles/CBMickFVX3lxTE5TT3RSLTJGdXVmMXRxVVUtakdWV2ZlRFpEa3BxREVmS3h6VEpGUm12X3RmZEJ6Nl9KaEtvVXJGLVd2Q3BJXzFZekZodG5ZUlp5NUtkdDhhd3NoLXZVclVDV1VuX0lTYWNOSHpVa1ZVWE9tQQ?oc=5) — *Google News / Latin Times*
- `11/12 06:10` [Gold Miners Post Strong Q2 Despite Price Pullback and Rising Costs](https://news.google.com/rss/articles/CBMinAFBVV95cUxQaG0tTkMyZWRDWkFwSGEyU01aTnBMN1FyaHp3bWZXOWZiLWcycE85eDNCVEo2SWN4QXZRZnJEbzRmYUI3R0dlNk5oXzE5cm84QU0xbnZKM3ItNTF2V2JtSWVGRk81cER4Ty1JNVZVUWEtU0pJUGdFOXAzcGJhUUJJY1U4Qk9SN3BHUVR1WGtqTnZ4aGpURmFoaXlQMUY?oc=5) — *Google News / IndexBox*
- `11/12 06:05` [Gold miners face rising costs and lower margins in Q2 2026 despite strong cash flow and high gold prices.](https://news.google.com/rss/articles/CBMingFBVV95cUxNczJvR0ptdEdEcVpVWlAzTU1HTFgySTZScTg4SHFKczNidXhNWEtwS0tCVjJpbjVqNXBPdl9mcFBmdjg5UzZDcHc0bHZKX0J0MlpDY19hQk5PLUszVjg5Y3FOQms3bXlvYm1NUDRxY2hFNFRvZ2puUGxqWjZRX01DOGtzczNCSFB0Vy1SdkZmcndYTU5JajJtU2wtclR0Zw?oc=5) — *Google News / Pluang*
- `11/12 06:00` [Gold falls below $4,000 as dollar surges, Fed hike bets rise - Kitco PM Report](https://news.google.com/rss/articles/CBMitwFBVV95cUxPQkRUZVFwS0pNb3NBQjg1aUNMcjQ5Y0pVU0NLUjU4M3pJLXh5WFYtd3ZFMXhzM2I2Y3RDd25WeXhvaWY4ZHh1TzhrRVJIRzRFT2JJRWE3Tkc2MkVTOFNSZnUzQTVTMFltZXl1cVZRSXdmR0h4ekJUdEhHYjNGTmpnbU5ZTnhyZzlJVGxlc3F2NUhCa3pxd1UtRnd1Yk1UblE2Y05TdmNOR0xMbjhyQkhLZ0hCUFJwYUE?oc=5) — *Google News / Kitco*
- `11/12 05:30` [Canada’s OR Royalties revenue rises 26% on higher gold prices](https://news.google.com/rss/articles/CBMiqgFBVV95cUxOQzBsOFo0aW91TlNOMWVxVmswZXlGZXRlcWJCTHFYalc4aXNUZ2hIYVJYYllIMWNsYjRPWlNJUkI0SURNSklpSVA0UzVyZ2RPVFpIMzVFODV6TUxJM1Nnd2xIRktLdXJyX0FtMVpidGZKRnc1UEFhb2gtQlY2VUt6cGxHNWNjQV9pNVFTcGFBVG04UnAwR1NVVXQtWUNXSENwVTYtN0hhUllNUQ?oc=5) — *Google News / Investing.com Canada*

### 原油
- `11/12 09:00` [Who Will Buy Sable’s Oil if California Keeps Closing Refineries?](https://oilprice.com/Energy/Crude-Oil/Who-Will-Buy-Sables-Oil-if-California-Keeps-Closing-Refineries.html) — *OilPrice*：Situated off California’s Santa Barbara coast, the Santa Ynez project operated by Sable Offshore has become a litmus test of President Trump’s determination to revive oil production in a state pursuin…
- `11/12 08:09` [Gas prices fall below $4 as more oil flows, but experts warn relief may not last](https://news.google.com/rss/articles/CBMiygFBVV95cUxPa0FSYmdOTVk0V1VoY2VtUmZURnRQbVZjczUtNzM3ZnJWRDBUNkx0LU1GWjVfVlNTTUxFSUprQVBJM29fOXhfNXdFTUQ2WnRhZ2tFR0pJQ3VscG5oM1BvUjhSME5NQV9SLUM5eUNJMnV1Y1J0aVJIaHlveURPTUQ1MVk4T05OQzV3ZzZ6X0xMcWRhUGZaRVNYRE9NckwyaldCQlhINDB4SXJjTF9IdEZFOUFCcGtzNWVjei1jeFlndS1UOTdTaUt0YV93?oc=5) — *Google News / Scripps News*
- `11/12 08:00` [Global Power Grid Expansion Surges 50% to Record High](https://oilprice.com/Energy/Energy-General/Global-Power-Grid-Expansion-Surges-50-to-Record-High.html) — *OilPrice*：Utilities built a total of over 21,000 km of new transmission lines last year, the International Renewable Energy Agency reported this week at COP31, citing data from a group called the Utilities for…
- `11/12 07:53` [Oil prices dip on report of more U.S.-Iran diplomacy plans](https://news.google.com/rss/articles/CBMisAFBVV95cUxQQnZvSlNXaWZKY2RaNkc4VlA1UlY0b1kwQmlFXy1BRHpJTFdfb05IY0hTbDI5TVJXMENhSEcxRmhCUnRCRERPMzZBY09FdXdoMk9UeFFLV3BQUWw5MnRFdGt3NEpmTmhiZkhHdDl4SEpGakU1YVhkVjEtZTZvNmpNdDFCYTdEcVRZaVk4aXJuaGd6VlJZNDlWa1VOSmkwZ3lrZ3lHZ1FpRVRXLWJoMThZcQ?oc=5) — *Google News / Investing.com*
- `11/12 07:06` [ASX Preview: Australian Shares to Fall as Oil Prices Rebound; Aristocrat Leisure Fiscal 2026 Adjusted Earnings Up, Revenue Down](https://news.google.com/rss/articles/CBMiiwFBVV95cUxNUWcyak1ua0pTUWZGSGVRSEhWS3NqUGZySFZnSm9lcUVZenMzNUFwSGhqUFJBTGRwZDlEbFNJU0NsLTBDUkhra2NzV0hEclZ2aXNvRTF5TUY2bV9KZlpmcE1RR0ZCcHRvWENYVFpPQklKdmI2UFRzeDRBbFVJNklUdjdOdkJ4NFZaZ1dZ?oc=5) — *Google News / Yahoo Finance Australia*
- `11/12 07:00` [Fusion's Next Race Is Over Superconducting Tape, Not Plasma](https://oilprice.com/Alternative-Energy/Nuclear-Power/Fusions-Next-Race-Is-Over-Superconducting-Tape-Not-Plasma.html) — *OilPrice*：The race to build the world’s first commercial fusion power plant has moved out of the physics lab and onto the factory floor. Private investors poured a record $4.48 billion into fusion last year, a…
- `11/12 06:19` [Stocks And Precious Metals Charts - Gold And Oil Prices Face Resistance - TalkMarkets](https://news.google.com/rss/articles/CBMiO0FVX3lxTE5OLVk4M1R3ZXlOM1htSmhVUmRIUjQ4NnNGZDByT2VjLVJEcHhNVmZJUU8yNE05T2tvSndB?oc=5) — *Google News / Howl.Link*
- `11/12 06:16` [World crude oil prices continue slide; LPG rollback seen](https://news.google.com/rss/articles/CBMirgFBVV95cUxObnpYZ19ZV2F1a0hEU1RvT2xsYnp0RG4tU01sN09Pa0Y4YVdRUWtvZ1NEUnJDSjBfN3k2Tmp6eUROSi01dDNYWmNpREJ1ZmVYWFliS2dXWnExQWVzSGxUTWdIWTE2ZXlGdWYtTUxRX2hZVW9hd3NJM2dnU1NBTF9BcEdJTF9STE5MUjYtRlE1dEhPYVN0MFlmcUg2UnB4dTd6bVFKQndSUWhOZFdMUEE?oc=5) — *Google News / ABS-CBN*

### 中文（鉅亨網美股）
- `11/12 09:10` [德國將停止使用中製無人機進行訓練](https://news.cnyes.com/news/id/6659886) — *鉅亨網*：德國政府稱，聯邦國防軍將停用中國製無人機訓練，改採德國或其他歐洲機型，以提升自給與技術自主。&#10;德國國防部要求部隊使用商用無人機前先做風險評估，避免涉及安全資訊外洩；尚未公布現役中國製無人機數量。&#10;德軍已要求停用中國商用無人機，憂演習資料、營區影像與武器系統資訊恐流向中國情報機構。
- `11/12 09:09` [再爆偷吃！黃仁勳回《紐時》記者一句秒掛電話](https://news.cnyes.com/news/id/6660002) — *鉅亨網*：《紐約時報》周三踢爆黃仁勳還曾與一名向他直接報告的高階主管發生婚外情。&#10;黃仁勳曾與輝達醫療事業副總裁Kimberly Powell發展關係。&#10;今年3月，黃仁勳被拍到與Powell在加州聖荷西一家飯店內親密互動的畫面。
- `11/12 09:05` [輝達盤後暴跌10%！黃仁勳再傳偷吃恐遭解職 日韓晶片股走勢分化 SK海力士死裡逃生翻紅 揖斐電跌近8%](https://news.cnyes.com/news/id/6660009) — *鉅亨網*：外媒爆料黃仁勳與醫療保健業務副總裁Kimberly Powell有染，輝達盤後股價一度重挫近10%。&#10;報導稱輝達董事會將於週四(12日)開會決定是否讓黃仁勳解職下台，並已找外部機構尋找潛在繼任者。&#10;亞洲晶片股走勢分歧；揖斐電、愛德萬測試下挫，SK海力士轉漲，市場關注AI算力龍頭後續影響。
- `11/12 08:40` [年砸1450億美元擴建AI基礎設施 Meta迎關鍵期換將：17年老臣交棒、AWS老將接手](https://news.cnyes.com/news/id/6659932) — *鉅亨網*：Meta基礎設施負責人Janardhan明年初離任，由7月加入的AWS老將David Brown接任並出任Meta Compute技術長。&#10;祖克柏規劃十年內建數十GW、長期擴至數百GW資料中心容量。&#10;Meta全年資本支出預測為1,350億至1,450億美元，第三季自由現金流為負21.2億美元。
- `11/12 08:30` [AI資料中心缺電 800V直流發電機搶手 Hyliion客戶洽談出資擴產 財測再上修](https://news.cnyes.com/news/id/6659996) — *鉅亨網*
- `11/12 08:10` [美股重點新聞摘要2026年11月12日](https://news.cnyes.com/news/id/6658463) — *鉅亨網*
- `11/12 08:00` [美股收盤／CoreWeave大漲24.7%振奮AI股費半漲5.7%、那指升1.6%](https://news.cnyes.com/news/id/6659941) — *鉅亨網*
- `11/12 07:59` [彭博：輝達董事會準備撤換黃仁勳 委外物色新執行長 盤後股價崩超7%](https://news.cnyes.com/news/id/6659928) — *鉅亨網*：傳《華爾街日報》指董事會擬撤換執行長黃仁勳，輝達股價盤後跌7.70%至160.35美元&#10;輝達週三收漲2.87%，但盤後急跌13.38美元，拖累那指100期貨跌1%&#10;CoreWeave財報優於預期帶動AI股反彈；市場續關注通膨、利率與美國就業、房市數據


## 8. Pre-market and early session（盤前與早盤）

In [8]:
# 只需要先跑第 1 格。
# ・台北傍晚～深夜執行：即時看當天美股的盤前與早盤。
# ・隔天清晨寫報告時執行：自動回看剛收盤那個交易日的盤前與早盤（寫「盤前 → 早盤 → 終場」用）。
SESSION_DATE = None                 # None＝自動；要回看某一天就填美東日期，例如 "2026-11-11"
PRE_WINDOW = ("04:00", "09:30")     # 盤前（美東）＝台北 17:00–22:30（冬令）／16:00–21:30（夏令）
EARLY_WINDOW = ("09:30", "12:00")   # 早盤（美東）＝台北 22:30–01:00（冬令）／21:30–00:00（夏令）
FUT = {"ES=F": "標普500期貨", "NQ=F": "那斯達克100期貨", "YM=F": "道瓊期貨"}
EARLY_IDX = {s: n for s, n in INDICES.items() if s != "^VIX"}
PRE_PATH = ["04:00", "06:00", "08:00", "08:30", "08:45", "09:30"]     # 08:30 是多數美國經濟數據公布時間
EARLY_PATH = ["09:35", "10:00", "10:30", "11:00", "11:30", "12:00"]   # 10:00 也常有數據（ISM、消費者信心）
# Finnhub 個股新聞（每檔 1 次請求）：{代號: 標題關鍵字}，標題沒提到公司的略過（Finnhub 會把不相干的新聞稿也掛在大型股底下）
STOCK_NEWS = {"NVDA": ("nvidia",), "AAPL": ("apple",), "MSFT": ("microsoft",), "AMZN": ("amazon", "aws"),
              "GOOGL": ("google", "alphabet"), "META": ("meta",), "TSLA": ("tesla",), "TSM": ("tsmc", "taiwan semi")}
STOCK_NEWS_PER_WINDOW = 2           # 每檔每個時段最多幾則
NEWS_PER_WINDOW = 30                # 綜合新聞每個時段最多幾則
CNBC_FEEDS = {"CNBC": 100003114, "CNBC Economy": 20910258, "CNBC Earnings": 15839135, "CNBC Finance": 10000664,
              "CNBC Tech": 19854910, "CNBC US": 15837362, "CNBC World": 100727362, "CNBC Energy": 19836768}


def bars(sym):
    """近 5 天的 5 分 K（美東時間）。期貨幾乎 24 小時都有，指數只有 09:30–16:00"""
    h = yf.Ticker(sym).history(period="5d", interval="5m", prepost=False)
    if h is None or h.empty:
        raise ValueError("Yahoo 沒有回傳 5 分 K")
    return h[["Open", "High", "Low", "Close"]].dropna().tz_convert(NY)


def cnbc_feed(name, fid):
    r = HTTP.get(f"https://www.cnbc.com/id/{fid}/device/rss/rss.html", timeout=TIMEOUT)
    r.raise_for_status()
    return [(datetime(*e.published_parsed[:6], tzinfo=timezone.utc), clean(e.get("title"), 300), e.get("link", ""),
             name, clean(e.get("summary")))
            for e in feedparser.parse(r.content).entries if e.get("published_parsed")]


def finnhub_items(label, data, words=()):
    return [(datetime.fromtimestamp(d["datetime"], timezone.utc), clean(d.get("headline"), 300), d.get("url", ""),
             f"{label} / {d.get('source', '')}", clean(d.get("summary")))
            for d in data if d.get("datetime") and d.get("headline") and not blocked(d.get("url"), d.get("source"))
            and (not words or re.search(r"\b(" + "|".join(words) + r")", d["headline"], re.I))]


def official_closes(symbols):
    """指數的官方日收盤（算「前收」用）；一次請求"""
    df = yf.download(symbols, period="10d", interval="1d", auto_adjust=False, progress=False)["Close"]
    return df.to_frame(symbols[0]) if isinstance(df, pd.Series) else df


now = pd.Timestamp.now(tz=NY)
if SESSION_DATE:
    S = pd.Timestamp(SESSION_DATE).date()
else:
    S = now.date()
    while S.weekday() >= 5:            # 週末：回看週五
        S -= timedelta(days=1)

# 行情（Yahoo）與新聞（CNBC、Finnhub）一次同時抓
tasks = {("bar", s): ("yahoo", partial(bars, s)) for s in [*FUT, *EARLY_IDX]}
tasks.update({("news", n): ("news", partial(cnbc_feed, n, fid)) for n, fid in CNBC_FEEDS.items()})
if FINNHUB_API_KEY:
    tasks["news", "Finnhub"] = ("finnhub", lambda: finnhub_items("Finnhub", finnhub("news", category="general")))
    for sym, words in STOCK_NEWS.items():
        tasks["stock", sym] = ("finnhub", partial(
            lambda sym, words: finnhub_items(sym, finnhub("company-news", symbol=sym, **{"from": f"{S:%Y-%m-%d}",
                                                                                           "to": f"{S:%Y-%m-%d}"}), words),
            sym, words))
tasks["daily", "指數日收盤"] = ("yahoo", partial(official_closes, list(EARLY_IDX)))
res = gather(tasks, {"yahoo": YAHOO_WORKERS, "news": len(CNBC_FEEDS), "finnhub": FINNHUB_WORKERS})
B = {k[1]: v[v.index <= now] for k, (v, err) in res.items() if k[0] == "bar" and v is not None}
DAILY = res["daily", "指數日收盤"][0]
errs = [f"{k[1]}：{err}" for k, (v, err) in res.items() if err]

cash_days = sorted({d for s in EARLY_IDX if s in B for d in B[s].index.date if d < S})
P = cash_days[-1] if cash_days else S - timedelta(days=3 if S.weekday() == 0 else 1)    # 前一個美股交易日
T = lambda d, hm: pd.Timestamp(f"{d} {hm}", tz=NY)
pre0, open_t, early1, ref_t = T(S, PRE_WINDOW[0]), T(S, PRE_WINDOW[1]), T(S, EARLY_WINDOW[1]), T(P, "16:00")
tw = lambda t: f"{t.tz_convert(TW):%H:%M}"
num = lambda v: f"{v:,.2f}"


def pct_at(df, ref, lo, t):
    """t 那一刻的價格（t 之前最後一根 5 分 K 的收盤）相對 ref 的漲跌；時間還沒到就留空"""
    v = df[(df.index >= lo) & (df.index < t)]["Close"]
    return "—" if t > now or v.empty else f"{(v.iloc[-1] / ref - 1) * 100:+.2f}%"


def path_table(names, rows, times):
    head = "| 台北時間 | " + " | ".join(f"{tw(T(S, t))}" for t in times) + " |"
    return "\n".join([head, "|:---|" + "---:|" * len(times)] + [f"| {names[s]} | " + " | ".join(r) + " |" for s, r in rows])


stamp = f"執行時間：台北 {now.tz_convert(TW):%m/%d %H:%M}＝美東 {now:%m/%d %H:%M}"
lines = [f"## 美股 {S:%m/%d}（{'一二三四五六日'[S.weekday()]}）盤前與早盤", f"{stamp}｜前一交易日：{P:%m/%d}", ""]

# ---- 盤前：指數期貨 ----
state = "尚未開始" if now < pre0 else f"進行中，數字截至執行時間" if now < open_t else "已結束，以下是開盤前最後價"
lines += [f"### 盤前｜指數期貨（台北 {tw(pre0)}–{tw(open_t)}，{state}）",
          f"漲跌基準：前一交易日美股收盤時（美東 {P:%m/%d} 16:00）的期貨價格，和現貨指數的漲跌可以直接比較。", "",
          "| 期貨 | 基準 | 開盤前 / 最新 | 漲跌 | 盤前高（台北時間） | 盤前低（台北時間） |", "|:---|---:|---:|---:|---:|---:|"]
rows = []
for s, name in FUT.items():
    f = B.get(s)
    base = f[f.index < ref_t]["Close"] if f is not None else []
    seg = f[(f.index >= ref_t) & (f.index < open_t)] if f is not None else []
    if not len(base) or not len(seg):
        lines.append(f"| {name} | — | — | — | — | — |")
        continue
    ref, last, w = base.iloc[-1], seg["Close"].iloc[-1], seg[seg.index >= pre0]
    hi = f"{num(w['High'].max())}（{tw(w['High'].idxmax())}）" if len(w) else "—"
    lo = f"{num(w['Low'].min())}（{tw(w['Low'].idxmin())}）" if len(w) else "—"
    lines.append(f"| {name}（{s}） | {num(ref)} | {num(last)} | {last - ref:+,.2f}（{(last / ref - 1) * 100:+.2f}%） | {hi} | {lo} |")
    rows.append((s, [pct_at(f, ref, ref_t, T(S, t)) for t in PRE_PATH]))
if any(v != "—" for _, r in rows for v in r):
    lines += ["", "盤前走勢（各時點相對基準的漲跌；台北時間）：", "", path_table(FUT, rows, PRE_PATH)]

# ---- 早盤：主要指數 5 分 K ----
state = f"尚未開盤（台北 {tw(open_t)} 開盤）" if now < open_t else "進行中，數字截至執行時間" if now < early1 else "已結束"
lines += ["", f"### 早盤｜主要指數（台北 {tw(open_t)}–{tw(early1)}，{state}）", ""]
rows, approx = [], []
if now >= open_t:
    lines += ["| 指數 | 前收 | 開盤（跳空） | 早盤高（台北時間） | 早盤低（台北時間） | 早盤結束 / 最新 |", "|:---|---:|---:|---:|---:|---:|"]
    for s, name in EARLY_IDX.items():
        d = B.get(s)
        prev = d[d.index.date == P]["Close"] if d is not None else []
        day = d[(d.index >= open_t) & (d.index < early1)] if d is not None else []
        if not len(prev) or not len(day):
            lines.append(f"| {name} | — | — | — | — | — |")
            continue
        off = DAILY[s].dropna() if DAILY is not None and s in DAILY else pd.Series(dtype=float)
        off = off[off.index.date == P]
        if off.empty:
            approx.append(name)                    # 抓不到官方收盤：退而用前一交易日最後一根 5 分 K
        ref, o, last = (off.iloc[-1] if len(off) else prev.iloc[-1]), day["Open"].iloc[0], day["Close"].iloc[-1]
        lines.append(f"| {name} | {num(ref)} | {num(o)}（{(o / ref - 1) * 100:+.2f}%） | "
                     f"{num(day['High'].max())}（{tw(day['High'].idxmax())}，{(day['High'].max() / ref - 1) * 100:+.2f}%） | "
                     f"{num(day['Low'].min())}（{tw(day['Low'].idxmin())}，{(day['Low'].min() / ref - 1) * 100:+.2f}%） | "
                     f"{num(last)}（{(last / ref - 1) * 100:+.2f}%） |")
        rows.append((s, [pct_at(d, ref, open_t, T(S, t)) for t in EARLY_PATH]))
    if rows:
        lines += ["", "早盤走勢（各時點相對前收的漲跌；台北時間）：", "", path_table(EARLY_IDX, rows, EARLY_PATH),
                  "", "前收是官方收盤價；高低點時間是該根 5 分 K 的起始時間。"
                  + (f"⚠️ {'、'.join(approx)} 抓不到官方收盤，前收改用前一交易日最後一根 5 分 K（有極小誤差）。" if approx else "")]

# ---- 新聞：依時間分到盤前、早盤 ----
seen, general, stock = set(), [], {}
for k, (items, err) in res.items():
    if k[0] in ("news", "stock"):
        for it in items or []:
            key = re.sub(r"\W+", "", it[1].lower())
            if key and key not in seen and not blocked(it[2]):
                seen.add(key)
                (general if k[0] == "news" else stock.setdefault(k[1], [])).append(it)


def news_lines(items):
    return [f"- `{t.astimezone(TW):%m/%d %H:%M}` [{title.replace('[', '(').replace(']', ')')}]({link}) — *{src}*"
            + (f"：{summary}" if summary else "") for t, title, link, src, summary in items]


for label, lo, hi in (("盤前", pre0, open_t), ("早盤", open_t, early1)):
    lines += ["", f"### {label}新聞（台北 {lo.tz_convert(TW):%m/%d %H:%M}–{hi.tz_convert(TW):%H:%M}）"]
    if now < lo:
        lines.append("- （時段尚未開始）")
        continue
    g = sorted(it for it in general if lo <= it[0] < hi)
    lines += news_lines(g[-NEWS_PER_WINDOW:]) or ["- （這個時段沒有 CNBC／Finnhub 綜合新聞）"]
    picks = [it for sym in STOCK_NEWS for it in sorted(i for i in stock.get(sym, []) if lo <= i[0] < hi)[-STOCK_NEWS_PER_WINDOW:]]
    if picks:
        lines += ["", f"**{label}個股新聞（Finnhub）**"] + news_lines(picks)
if not FINNHUB_API_KEY:
    lines += ["", "（未設定 FINNHUB_API_KEY：只有 CNBC 新聞）"]
if errs:
    lines += ["", "⚠️ 無法取得：" + "、".join(errs)]
display(Markdown("\n".join(lines)))

## 美股 11/11（三）盤前與早盤
執行時間：台北 11/12 09:36＝美東 11/11 20:36｜前一交易日：11/10

### 盤前｜指數期貨（台北 17:00–22:30，已結束，以下是開盤前最後價）
漲跌基準：前一交易日美股收盤時（美東 11/10 16:00）的期貨價格，和現貨指數的漲跌可以直接比較。

| 期貨 | 基準 | 開盤前 / 最新 | 漲跌 | 盤前高（台北時間） | 盤前低（台北時間） |
|:---|---:|---:|---:|---:|---:|
| 標普500期貨（ES=F） | 7,544.75 | 7,612.75 | +68.00（+0.90%） | 7,620.00（21:15） | 7,587.25（17:15） |
| 那斯達克100期貨（NQ=F） | 27,835.25 | 28,352.00 | +516.75（+1.86%） | 28,406.25（21:25） | 28,224.50（17:15） |
| 道瓊期貨（YM=F） | 52,317.00 | 52,424.00 | +107.00（+0.20%） | 52,501.00（21:55） | 52,259.00（18:10） |

盤前走勢（各時點相對基準的漲跌；台北時間）：

| 台北時間 | 17:00 | 19:00 | 21:00 | 21:30 | 21:45 | 22:30 |
|:---|---:|---:|---:|---:|---:|---:|
| 標普500期貨 | +0.71% | +0.77% | +0.85% | +0.96% | +0.92% | +0.90% |
| 那斯達克100期貨 | +1.62% | +1.76% | +1.87% | +1.96% | +1.87% | +1.86% |
| 道瓊期貨 | +0.23% | +0.03% | +0.17% | +0.28% | +0.27% | +0.20% |

### 早盤｜主要指數（台北 22:30–01:00，已結束）

| 指數 | 前收 | 開盤（跳空） | 早盤高（台北時間） | 早盤低（台北時間） | 早盤結束 / 最新 |
|:---|---:|---:|---:|---:|---:|
| 道瓊 | 52,231.89 | 52,196.35（-0.07%） | 52,338.10（22:30，+0.20%） | 52,055.14（23:20，-0.34%） | 52,306.98（+0.14%） |
| 標普 | 7,520.93 | 7,586.65（+0.87%） | 7,624.70（00:55，+1.38%） | 7,571.88（22:45，+0.68%） | 7,623.70（+1.37%） |
| 納指 | 25,065.87 | 25,452.79（+1.54%） | 25,586.82（00:55，+2.08%） | 25,327.23（22:45，+1.04%） | 25,582.40（+2.06%） |
| 費半 | 9,998.33 | 10,469.04（+4.71%） | 10,745.82（00:55，+7.48%） | 10,287.84（22:45，+2.90%） | 10,739.54（+7.41%） |

早盤走勢（各時點相對前收的漲跌；台北時間）：

| 台北時間 | 22:35 | 23:00 | 23:30 | 00:00 | 00:30 | 01:00 |
|:---|---:|---:|---:|---:|---:|---:|
| 道瓊 | -0.10% | -0.04% | -0.26% | -0.12% | +0.04% | +0.14% |
| 標普 | +0.90% | +0.87% | +0.91% | +1.18% | +1.26% | +1.37% |
| 納指 | +1.56% | +1.30% | +1.36% | +1.89% | +1.95% | +2.06% |
| 費半 | +4.19% | +4.13% | +5.21% | +6.65% | +6.84% | +7.41% |

前收是官方收盤價；高低點時間是該根 5 分 K 的起始時間。

### 盤前新聞（台北 11/11 17:00–22:30）
- `11/11 18:06` [American soldier arrested on suspicion of spying for Russia](https://www.cnbc.com/2026/11/11/us-army-soldier-russia-espionage-arrest.html) — *CNBC*：The 41-year-old man is being detained in London under the U.K.'s National Security Act.
- `11/11 19:56` [CoreWeave provides silver lining for neoclouds as Nvidia storm swirls](https://www.cnbc.com/2026/11/11/coreweave-provides-silver-lining-for-neoclouds-as-nvidia-storm-swirls.html) — *CNBC Tech*：Huang's close ties to the White House are in the firing line
- `11/11 20:27` [Consumers are showing signs of strain even as they keep spending, CNBC/NRF Retail Monitor shows](https://www.cnbc.com/2026/11/11/october-retail-sales-cnbc-nrf-2026.html) — *CNBC Economy*：Retail sales rose at their slowest pace in nearly a year in October as American consumers showed signs of strain from high prices.
- `11/11 20:38` [Mortgage rates jump sharply higher, after easing for four straight weeks. Here's why](https://www.cnbc.com/2026/11/11/mortgage-rates-jump-sharply-higher-after-easing-for-4-straight-weeks.html) — *CNBC US*：After falling for a month, mortgage rates jumped sharply higher on a hotter-than-expected inflation report.
- `11/11 20:42` [Rep. Jim Jordan, Sen. John Thune elected GOP leaders in Congress](https://www.cnbc.com/2026/11/10/jim-jordan-congress-republicans-midterm.html) — *CNBC*：House Speaker Mike Johnson, R-La., announced the day after Election Day he would not run for minority leader in the 120th Congress.
- `11/11 20:57` [Hundreds of schools in Italy, UK evacuated over 'hoax' bomb threats](https://www.cnbc.com/2026/11/11/hoax-bomb-threats-schools-italy-uk.html) — *CNBC*：Hundreds of schools across Europe were evacuated on Wednesday after receiving bomb threats, police forces say, amid growing worries of hybrid threats.
- `11/11 21:18` [Nissan cuts sales forecast again on China woes as CEO outlines plan to fight back](https://www.cnbc.com/2026/11/11/nissan-sales-forecast-earnings-china.html) — *CNBC*：Intensifying competition in the world's largest auto market forced Nissan to cut its annual sales forecast for a second time in months.
- `11/11 21:46` [Nvidia's Huang tries to project 'business as usual' in return to office after affair revelation](https://www.cnbc.com/2026/11/10/nvidia-jensen-huang-affair-office.html) — *CNBC Tech*：Nvidia employees described the mood as "sad" as CEO Jensen Huang returned to the office after disclosing an affair.
- `11/11 22:13` [Jim Cramer's top 10 things to watch in the stock market Wednesday](https://www.cnbc.com/investingclub/2026/11/11/jim-cramers-top-10-things-to-watch-in-the-stock-market-wednesday.html) — *Finnhub / CNBC*：CoreWeave lifts the AI trade, and my advice to Nvidia.
- `11/11 22:13` [Zelenskyy warns Ukraine is 'not real estate' as experts say a Russia peace deal remains elusive](https://www.cnbc.com/2026/11/11/ukraine-russia-peace-talks-zelenskyy-putin.html) — *CNBC*：President Zelenskyy warned Tuesday that his country "was not real estate," as a second and final day of U.S.-brokered peace talks got underway.

**盤前個股新聞（Finnhub）**
- `11/11 21:51` [Update: US Equity Futures Rise Pre-Bell With Nvidia, CoreWeave Gaining Amid Tech Sector Rebound](https://finnhub.io/api/news?id=86fda81b0d9d6a970df1acd9e01fb8ff7a3a1fb77259b5fcd7451013c44fb9c7) — *NVDA / Yahoo*：(Updates with economic data, recent oil price movement, world markets' overview and corporate stock
- `11/11 21:54` [Nvidia stock rebounds, chip stocks surge as CoreWeave earnings boost AI trade](https://finnhub.io/api/news?id=10fa1c062c0a739b3613bdd82c5ec00f0685011cab9e0271f638ec29e5521d06) — *NVDA / Yahoo*：Nvidia stock popped on Wednesday, snapping a two-day losing streak.
- `11/11 20:57` [Buy, Sell, or Hold? Wednesday’s Top Upgrades and Downgrades: Akamai Technologies, Apple, BILL Holdings, Corning, Form Factor, Guidewire Software, NVIDIA, Riot Platforms, Toast, and More](https://finnhub.io/api/news?id=7cdf40837326d672aab7d0eada0eb35580bdf7937b88127cfefe5e202c35ae1b) — *AAPL / Yahoo*：Hot inflation data rattled Tuesday's close, and now one chip giant faces a Baird downgrade tied to concerns about its CEO. Which analyst calls, from Apple's upgrade to a steep Akamai cut, deserve your…
- `11/11 21:03` [How Much Capital Gains Tax Does a 61-Year-Old Owe on $750,000 of Inherited Apple Stock?](https://finnhub.io/api/news?id=7eb80ecbdd062b84cd92b8ac4eb1586583d833dfbadeb2d4f8dcc22a754d79fd) — *AAPL / Yahoo*：Apple has soared nearly 1,300% in a decade, and one retiree assumes the IRS will claim a huge slice of her inheritance. A single tax rule rewrites that math, but a lifetime gift would have flipped the…
- `11/11 20:00` [Northern Trust leans on Microsoft FDEs to boost agentic buildout](https://finnhub.io/api/news?id=2b630d3e2de4da3ede3af91eaad05a3e28695f283c189ac7defaad7189b837dc) — *MSFT / Yahoo*：The financial services firm cut analyst’s data preparation time by 83% with the use of an onboarding agent, the companies said.
- `11/11 20:13` [Microsoft Stock Fell After Anthropic Launches Claude Sheets to Rival Excel](https://finnhub.io/api/news?id=b9e4b92fee3f1ace7785f0e1d29b26ee3b38423d6cd77fe6578dd4a3df9abcc3) — *MSFT / Yahoo*：Microsoft shares fell about 2% after Anthropic launched AI tools for spreadsheets, whiteboards and data dashboards, intensifying competition with Microsoft 365 and its Copilot assistant.
- `11/11 19:01` [ETFs Are Buying Amazon (AMZN) on Monday](https://finnhub.io/api/news?id=0f0911ce19c48bf41832af0f94189a2276b6bf52d91062be500dee7031154938) — *AMZN / Yahoo*：ETFs bought $412.9 million of Amazon stock on Monday, Nov. 9.
- `11/11 20:15` [Why Alphabet and Amazon Will Always Need Nvidia](https://finnhub.io/api/news?id=25d5650df933cc5a8580a39961454bc061417fbe4fc8ea0a988cc94b1b202479) — *AMZN / Yahoo*：Google sells its own TPUs and Amazon pushes Trainium, yet both keep writing NVIDIA enormous checks. One supply bottleneck and one software moat explain why the custom-chip threat may be harder to exec…
- `11/11 20:35` [Alphabet or CoreWeave: Which AI Stock Is Better to Buy in a Market Crash?](https://finnhub.io/api/news?id=9952f9b7ba197523596b4b1448966a52b67a59d5cc018a3c908070dc8084bba9) — *GOOGL / Yahoo*：Alphabet and CoreWeave are both positioned to benefit from AI, but one has a much stronger financial foundation.
- `11/11 21:17` [Google Now Has Two of the Top AI Models for Accounting Work, DualEntry Benchmark Finds](https://finnhub.io/api/news?id=0a40df70a73df502d3daa38bcf929f3c16e3c361df2d3545044b5beff7ad9481) — *GOOGL / Yahoo*：NEW YORK, November 11, 2026--Google's newly released Gemini 4 Carbon scored 88.1% on the DualEntry Accounting AI Benchmark, tying for second among 24 leading AI models tested. Google's Gemini 4 Argon…
- `11/11 19:04` [State Street Comm Services SPDR (XLC) Sells $29.9 Million of Meta on Monday](https://finnhub.io/api/news?id=8bc6a80a3287fc7684867f88cd30cd338ec7eacfbd39119fd67309cc2627cc2b) — *META / Yahoo*：The $22.12 billion fund sold $139.5 million of stock and bought nothing, cutting all 22 of its positions.
- `11/11 21:24` [Meta Fined by Brazil Electoral Court for Allegedly Failing to Provide Equal Content During Presidential Campaign](https://finnhub.io/api/news?id=4c51f312537b484e6acd09cf2a51c496b86a78f396a53c1808d9d20a6e979276) — *META / Yahoo*：Meta Platforms' (META) Facebook was fined 7.5 million Brazilian reais ($1.34 million) by Brazil's Su
- `11/11 20:12` [Tesla wins major European approval with a surprising catch](https://finnhub.io/api/news?id=510ad94bb96127a51083a7f99800bb485b7e82f557d5c0eacd4f20f53fa32014) — *TSLA / Yahoo*：The decision allows Tesla to offer its supervised driving technology to Finnish customers while broader European Union approval remains pending.
- `11/11 22:20` [How Much Is Tesla’s Robotaxi Actually Worth?](https://finnhub.io/api/news?id=b07bda4fe1d6414a7c54c85a567ec87bc7186a87bcdd54d59725d11968cf56b9) — *TSLA / Yahoo*：Tesla's Austin geofence just grew, unsupervised miles are surging, and a $1.5 trillion valuation leans heavily on the story. But Robotaxi has no reported revenue yet, so one future earnings report may…
- `11/11 21:28` [TSMC plans up to $1bn investment in Anthropic, OpenAI](https://finnhub.io/api/news?id=0598aa17c435cc9df026fff821e7d70ac548cd8e76be3bff6ff8ce616b0e3c5b) — *TSM / Yahoo*：TSMC's board approved an investment of $500m to acquire Anthropic common shares at the company’s IPO price.

### 早盤新聞（台北 11/11 22:30–01:00）
- `11/11 22:39` [Jensen Huang's affair with alleged Chinese spy further complicates Nvidia’s China problem](https://www.cnbc.com/2026/11/10/jensen-huang-nvidia-china-ties.html) — *CNBC Tech*：The allegations against Jensen Huang have brought new attention to the Nvidia CEO's potential influence over U.S. policy on chip sales to China.
- `11/11 23:00` [JetBlue's CEO has pushed back on creditor fears and is betting on premium to return to profitability](https://www.cnbc.com/2026/11/11/jetblue-ceo-premium.html) — *Finnhub / CNBC*：JetBlue will give reporters a look at its new first class seats on Wednesday.
- `11/11 23:14` [Measles is surging in the U.S. — and outbreaks can cost millions](https://www.cnbc.com/2026/11/11/measles-outbreak-us-economic-cost.html) — *CNBC*：A new tool developed by Johns Hopkins researchers estimates that the 2025 measles outbreak in the U.S. carried an economic cost of more than $37 million.
- `11/11 23:20` [WNBA Finals MVP Gabby Williams: 'Our existence is political'](https://www.cnbc.com/2026/11/11/gabby-williams-embraces-role-in-wnbas-political-and-cultural-debates-.html) — *CNBC*：Golden State Valkyries star Gabby Williams speaks about winning the WNBA championship, being named Finals MVP, and the culture war over women's sports.
- `11/11 23:33` [FDA warns of vision loss risk linked to popular GLP-1 drugs like Wegovy, Zepbound](https://www.cnbc.com/2026/11/10/fda-warning-vision-loss-glp1-wegovy-zepbound.html) — *CNBC US*：The agency has found evidence of a potential increased risk of a rare eye condition linked to the use of GLP-1 drugs used for weight management and diabetes.
- `11/11 23:53` [CoreWeave CEO Intrator tells CNBC he has canceled his planned share sale program](https://www.cnbc.com/2026/11/11/coreweave-ceo-election-data-center-buildout.html) — *CNBC Earnings*：After CoreWeave beat on revenue and upped its forecast, CEO Mike Intrator indicated the company will be fine despite last week's Democratic midterm sweep.
- `11/12 00:05` [Tired of spending time fixing other people's AI slop at work? How to talk to them about it](https://www.cnbc.com/make-it/2026/11/11/push-back-coworker-boss-extra-work-ai.html) — *CNBC*：Executive coach Melody Wilding shares four ways to address the extra work landing on your plate without the conversation becoming confrontational.
- `11/12 00:49` [Fed's Goolsbee calls inflation report 'borderline disturbing' as central bank weighs more hikes](https://www.cnbc.com/2026/11/11/goolsbee-october-inflation-fed-rates-disturbing-distressing.html) — *CNBC Finance*：The consumer price index rose more than expected in October, raising alarm among Federal Reserve officials.
- `11/12 00:51` [Schumer accuses Trump of defending Nvidia CEO Jensen Huang at cost of national security](https://www.cnbc.com/2026/11/10/moolenaar-nvidia-jensen-huang-affair-chinese-spy.html) — *CNBC Tech*：Rep. John Moolenaar, R-Mich., the Republican chairman of the Select Committee on China, told CNBC he expects answers about Jensen Huang's relationship.
- `11/12 00:59` [Jim Cramer's plan for our Intel position. Plus, a rotation loser to buy here](https://www.cnbc.com/investingclub/2026/11/11/jim-cramers-plan-for-our-intel-position-plus-a-rotation-loser-to-buy-here.html) — *Finnhub / CNBC*：The Investing Club holds its "Morning Meeting" every weekday at 10:20 a.m. ET.
- `11/12 00:59` [Bessent moved to military base amid 'growing threats' from Iran](https://www.cnbc.com/2026/11/11/us-iran-war-trump-hormuz-bessent-threats.html) — *CNBC World*：Bessent's role leading Operation Economic Outcast has made him one of the Trump administration's top targets of Iranian threats, The New York Times reported.

**早盤個股新聞（Finnhub）**
- `11/11 23:50` [Prediction: After Losing Over $1.7 Trillion in Market Cap This Year, Nvidia Stock Will Rebound in Epic Fashion. Here's Why.](https://finnhub.io/api/news?id=09dd8df95b49c62d81cda11024a15e994906042d7ccd1b76cb04feebdd8dc9aa) — *NVDA / Yahoo*：Nvidia stock has crashed 27% over the last month.
- `11/11 23:53` [Wall Street is cooling on Jensen Huang's Nvidia — but history says don't give up on the stock just yet](https://finnhub.io/api/news?id=8f33137fad56b117a727fd73024c2072cbebdeb8752f81ddf97d8c01807c2f3a) — *NVDA / Yahoo*：Morgan Stanley and Bank of America pulled Nvidia from their best-ideas lists, but nearly all analysts still rate the stock a Buy.
- `11/11 22:52` [Stock Market Today (Nov. 11, 2026): Nasdaq, S&P 500 gain after CoreWeave earnings beat; Apple hits new record](https://finnhub.io/api/news?id=3aa63f4e93691d6ce01bd0796b32d29265849fada10cc82dbb1a311c3a065f52) — *AAPL / Yahoo*：CoreWeave soars after the cloud provider reported better-than-expected quarterly revenue and raised its revenue outlook.
- `11/11 23:27` [Why the iPhone Duo Could Kick Off a ‘New Decade’ for Apple](https://finnhub.io/api/news?id=c9d8d94a905cf44630bf97b13336927b6252a449bb07179fb8bedcb9c93e0a97) — *AAPL / Yahoo*：Apple stock is having its best year since 2024. One firm believes the iPhone Duo could unlock a decade of potential.
- `11/12 00:00` [Sequoia Backs Microsoft Veterans' Edelbyte with $70M to Challenge Data Infrastructure Giants](https://finnhub.io/api/news?id=f958c34087dfbe67266997e4c6b9f78f62a91986468cb8f70b243c064a98637f) — *MSFT / Yahoo*：Four Microsoft veterans are targeting the $300B data market currently dominated by Snowflake, Databricks, and ClickHouse. Roughly a year after founding , the team has already secured leading enterpris…
- `11/12 00:54` [Microsoft Stocks Decline 2.5% as $1 Billion Anthropic Stake Tests Partner-Rival Balance](https://finnhub.io/api/news?id=44efdaf6a59686b8377175b41092bd1398aacceb048312ab6735fbd3c333e39b) — *MSFT / Yahoo*：Microsoft joins Anthropic's IPO placement while Azure collects fees on Claude sales under joint go-to-market arrangements.
- `11/12 00:00` [Will Rising AI Capex Weigh on Amazon's Free Cash Flow Prospects?](https://finnhub.io/api/news?id=de65b108c8f0f3c249c6a33385c933509c744132e71a285d568f22fcf2d5db37) — *AMZN / Yahoo*：AMZN's $220B AI capex plan is squeezing free cash flow despite surging AWS growth and stronger operating profits. Can its cloud momentum offset the strain?
- `11/12 00:35` [A Major Catalyst Could Send Amazon Stock to New Highs](https://finnhub.io/api/news?id=0019f903aa11dd59a66b3e5f43405c1e6c1d569763ee706d21ca00f10a047283) — *AMZN / Yahoo*：Amazon just posted its fastest cloud growth in nearly five years, yet a $220 billion spending binge has pushed free cash flow negative. Which force wins out, and how far could the stock run if the bul…
- `11/11 23:30` [Alphabet explores first Hong Kong and Singapore dollar bond sales worth up to $3 billion](https://finnhub.io/api/news?id=8552e50318542b49a04d15c40e77c496333b21cf4ce249aa899aacedf9acc47e) — *GOOGL / Yahoo*：Alphabet Inc (NASDAQ:GOOG), the Google parent company, is exploring its first Hong Kong and Singapore dollar bond sales worth up to $3 billion combined. The technology group is targeting between $1 bi…
- `11/11 23:55` [Eric Schmidt, Former Google CEO, Should Run Nvidia](https://finnhub.io/api/news?id=a564b5c6bbc0a10b556a2c347633ce2106c449372e431daf2a17f8542acbf87a) — *GOOGL / Yahoo*：A law firm probe into Jensen Huang's ties to Alice Huang could end his run at Nvidia, and the company has no obvious heir. Who could Wall Street, governments, and customers all accept in his place?
- `11/12 00:24` [Reddit Rides on Strong Ad Growth: Can It Outpace META and SNAP?](https://finnhub.io/api/news?id=76fb9adb18675ffd961a8a47fe2fd2fb727d5b600abdc1b98e72fcc13cafca1d) — *META / Yahoo*：RDDT's surging ad revenues, AI-powered targeting, and expanding advertiser base fuel growth as it battles Meta Platforms and Snap for digital ad dominance.
- `11/12 00:56` [Meta infrastructure chief Santosh Janardhan to step down in early 2027 - Bloomberg](https://finnhub.io/api/news?id=1a95d92ee8b13672c264ab22a132f03965f71e07ce7438670191ee463630f1b3) — *META / Yahoo*：Investing.com -- Meta Platforms Inc's (NASDAQ:META) longtime infrastructure chief and engineering co-head Santosh Janardhan will step down in early 2027, marking a pivotal executive transition as the…
- `11/11 23:45` [Elon Musk Warns Tesla May Have to Refund Cybertruck Deposits as Production Winds Down](https://finnhub.io/api/news?id=963b19ad86e4bff5221b10a13042ddff8a52fcb81f9f9c73b7ebe0b7b3cc5916) — *TSLA / Yahoo*：Some Tesla customers hoping to get a new cybertruck are going to be disappointed. CEO Elon Musk says Tesla is winding down production of the popular car at its Texas factory next year and there won’t…
- `11/12 00:40` [Tesla Clears Major Hurdle in European Self-Driving Push](https://finnhub.io/api/news?id=c90f3aafb63a339eeea0f92c9335be6408ef3af5ef8e5547a6c888539d141f28) — *TSLA / Yahoo*：A small regulatory victory could unlock much bigger opportunities.
- `11/12 00:07` [Something happened in October that changes the TSMC stock debate](https://finnhub.io/api/news?id=3aa25be6a3c9a02d5222bdc72aa0b37ee5b29a4e68c42cc89f29c2423bb6717d) — *TSM / Yahoo*：TSMC didn’t issue guidance with its October numbers because one number did all the talking.
- `11/12 00:55` [TSMC Stocks Climb 3.37% After Record NT$586.9 Billion October Revenue](https://finnhub.io/api/news?id=aa13a6525794366c993ea8409ee504ee238ccd6f794bf44b0b1af6908363e001) — *TSM / Yahoo*：October sales jumped 59.7% year over year, and TSMC's investment arm joins Anthropic's IPO placement with $500 million.